### ~

In [1]:

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#
from pathlib import Path
import shutil
import subprocess
import builtins



ROOT = Path.cwd()

def here():
    return ROOT

def make_dir(path):
    p = ROOT / path
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_file(path, content=""):
    p = ROOT / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return p

def remove_path(path):
    p = ROOT / path
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()
    return p

def run_cmd(*cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode
#
import os
from pathlib import Path
import shutil
import subprocess


class WorkspaceBase:
    @staticmethod
    def get_notebook_name():
        notebook_files = [f for f in os.listdir('.') if f.endswith('.ipynb')]
        if notebook_files:
            thienotebooktitleworkspace = os.path.splitext(notebook_files[0])[0]
        else:
            thienotebooktitleworkspace = "Untitled Notebook"
        return thienotebooktitleworkspace

    def get_root(self):
        return self.root

    def make_folder(self, name):
        folder = self.root / name
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def write_text(self, path, text):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        return path

    def read_text(self, path):
        return Path(path).read_text(encoding="utf-8")

    def run_cmd(self, *args):
        result = subprocess.run(args, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)

    def __init__(self, root=None):
        if root is None:
            root = f"{self.get_notebook_name()}_workspace"
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

class Problem_(WorkspaceBase):
    def __init__(self, name, files=None, root=None):
        super().__init__(root=root)
        self.name = name
        self.files = files or {}
        self.folder = self.make_folder(self.safe_name())

    def safe_name(self):
        return self.name.strip().lower().replace(" ", "_")

    def add_file(self, relative_path, content):
        if content and content[0] == '\n':
            content = content[1:]
        self.files[str(relative_path)] = content

    def get_file(self, relative_path):
        return self.files.get(str(relative_path))

    def remove_file(self, relative_path):
        self.files.pop(str(relative_path), None)

    def save(self, clear_first=False):
        if clear_first:
            self.reset_folder(self.folder)
        for relative_path, content in self.files.items():
            full_path = self.folder / relative_path
            self.write_text(full_path, content)

    def run(self, command, cwd=None, inputs=None):
        """Run a shell command. Pass inputs as a list of strings for stdin."""
        if cwd is None:
            cwd = self.folder
        stdin_text = "\n".join(str(v) for v in inputs) + "\n" if inputs else None
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True,
                                shell=True, input=stdin_text)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)

    def reset_folder(self, folder):
        folder = Path(folder)
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def load(self):
        loaded = {}
        for relative_path in self.list_all_files(self.folder):
            full_path = self.folder / relative_path
            loaded[relative_path] = self.read_text(full_path)
        self.files = loaded
        return loaded

    def summary(self):
        return {
            "name": self.name,
            "folder": str(self.folder),
            "file_count": len(self.files),
            "files": sorted(self.files.keys())
        }

    def __repr__(self):
        return f"Problem(name={self.name!r}, files={len(self.files)}, folder={str(self.folder)!r})"

class Problem(Problem_):
    def __init__(self, name, files=None, root=None):
        super().__init__(name, files, root)
        self.reset_folder(self.folder)

    def runpy(self, command, cwd=None, inputs=None):
        """Run a python file. Pass inputs as a list of strings for stdin.
        Example: _.runpy("main.py", inputs=["8", "3"])
        """
        print(f"Ran command: python {command} in {cwd or self.folder}")
        self.run(f"python {command}", cwd, inputs=inputs)

    def create(self, relative_path, content):
        self.add_file(relative_path, content)
        self.save()
        print(f"Created file: {relative_path} with content:\n{content}")

    def create2(self, relative_path, content, inputs=None):
        self.create(relative_path, content)
        self.runpy(relative_path, cwd=self.folder, inputs=inputs)

    def answer(self, content):
        self.add_file("ANSWER.md", content)
        self.save()
        print(f"Saved answer to ANSWER.md:\n{content}")

    def q(self, text):
        """Display the question/prompt text. Strips leading/trailing whitespace."""
        print(text.strip())
        return self

    def parse(self, text):
        """
        Mini-DSL to create files and run commands from one string.

        Syntax:
            Classic:
                F:
                <file content goes here>
                F:: filename.py          <- saves the content block to filename.py

            Reversed:
                F:: filename.py
                <file content goes here>
                F:                       <- closes and saves

            End-only:
                <file content goes here>
                F:: filename.py

            Alternate filename marker:
                FF: filename.py

            Run commands:
                R: main.py               <- runs main.py with no input
                R: main.py ["Eve", "5"]  <- runs main.py with those inputs

        Example:
            _.parse('''
            F:
            def greet(): return "hi"
            F:: greet.py

            F:
            import greet
            print(greet.greet())
            F:: main.py

            R: main.py
            ''')
        """
        import ast as _ast

        def _filename_from_marker(stripped_line):
            if stripped_line.startswith('F::'):
                return stripped_line[3:].strip()
            if stripped_line.startswith('FF:'):
                return stripped_line[3:].strip()
            if stripped_line.startswith('FILE::'):
                return stripped_line[6:].strip()
            return None

        lines = text.split('\n')
        collecting = False
        buffer = []
        current_filename = None
        mode = None  # "normal" (F: ... F::name) or "reversed" (F::name ... F:)
        loose_buffer = []

        for line in lines:
            stripped = line.strip()

            marker_filename = _filename_from_marker(stripped)

            if stripped == 'F:':
                if collecting and mode == "reversed" and current_filename:
                    self.create(current_filename, '\n'.join(buffer))
                    collecting = False
                    buffer = []
                    current_filename = None
                    mode = None
                else:
                    collecting = True
                    buffer = []
                    current_filename = None
                    mode = "normal"

            elif marker_filename is not None:
                if collecting and mode == "normal":
                    self.create(marker_filename, '\n'.join(buffer))
                    collecting = False
                    buffer = []
                    current_filename = None
                    mode = None
                elif not collecting:
                    if loose_buffer:
                        self.create(marker_filename, '\n'.join(loose_buffer))
                        loose_buffer = []
                    else:
                        collecting = True
                        buffer = []
                        current_filename = marker_filename
                        mode = "reversed"
                else:
                    self.create(marker_filename, '\n'.join(buffer))
                    collecting = False
                    buffer = []
                    current_filename = None
                    mode = None

            elif stripped.startswith('R:'):
                rest = stripped[2:].strip()
                # Split off optional inputs list at the end, e.g.  main.py ["a","b"]
                inputs = None
                bracket_idx = rest.find('[')
                if bracket_idx != -1:
                    filename = rest[:bracket_idx].strip()
                    try:
                        inputs = _ast.literal_eval(rest[bracket_idx:])
                        inputs = [str(x) for x in inputs]
                    except Exception:
                        inputs = None
                else:
                    filename = rest
                if filename:
                    self.runpy(filename, inputs=inputs)

            elif collecting:
                buffer.append(line)

            elif stripped:
                loose_buffer.append(line)

        if collecting and mode == "reversed" and current_filename:
            self.create(current_filename, '\n'.join(buffer))

        return self

#
def setin(*inputs):
    """
    A helper function to set test inputs for the input() function.
    Usage:
    setin("input1", "input2", "input3")
    This will set up the input() function to return "input1" on the first call,
    "input2" on the second call, and so on.
    To reset to normal input behavior, call setin() with no arguments or None:
    setin()
    """
    if inputs:
        input_iter = iter(inputs)
        def mock_input(prompt=""):
            try:
                value = next(input_iter)
                print(f"{prompt}{value}")
                return value
            except StopIteration:
                raise EOFError("No more inputs for testing")
        builtins.input = mock_input
    else:
        builtins.input = builtins._original_input_backup
              
#


### ~

In [76]:
'''11.1 Modules
Module basics
The interactive Python interpreter provides the most basic way to execute Python code. However, all of the defined variables, functions, classes, etc., are lost when a programmer closes the interpreter. Thus, a programmer will typically write Python code in a file, and then pass that file as input to the interpreter. Such a file is called a script.

A programmer may find themselves writing the same function over and over again in multiple scripts, or creating very long and difficult-to-maintain scripts. A solution is to use a module, which is a file containing Python code that can be imported and used by scripts, other modules, or the interactive interpreter. To import a module means to execute the code contained by the module and make the definitions within that module available for use by the importing program.

participation activity
11.1.1: A module is a file containing Python statements and definitions that can be used by other Python sources.


1

2

The functions can instead be defined in another file. The file can be imported as a "module".
def fct():
   # ...
def sq():   
   # ...
x = fct() * sq()
# ...
script1.py
def fct():
   # ...
def sq():   
   # ...
x = fct() / sq()
# ...
script2.pyfuncs.py
def fct():
   # ...
def sq():   
   # ...
script1.py
import funcs
script2.py
import funcs
x = funcs.fct() * 
      funcs.sq()
x = funcs.fct() / 
      funcs.sq()
fct()sq()
*
fct()sq()
/
def fct():
   # ...
def sq():   
   # ...
Static figure: Begin Python code (script1.py, before using a module): def fct(): # ... def sq(): # ... x = fct() * sq() # ... End Python code. Begin Python code (script2.py, before using a module): def fct(): # ... def sq(): # ... x = fct() / sq() # ... End Python code. Begin Python code (funcs.py module): def fct(): # ... def sq(): # ... End Python code. Begin Python code (script1.py, after using a module): import funcs x = funcs.fct() * funcs.sq() End Python code. Begin Python code (script2.py, after using a module): import funcs x = funcs.fct() / funcs.sq() End Python code. Step 1: A programmer writes scripts containing functions and code using those functions. Multiple scripts might define the same functions. script1.py and script2.py are shown defining the same functions fct() and sq(). Step 2: The functions can instead be defined in another file. The file can be imported as a "module". The module funcs.py is created with the same function definitions in both script1.py and script2.py. script1.py is rewritten to instead import the functions from the funcs.py module. script1.py is executed, starting with the import statement. To execute the calculation for x, the function names from the module are shown above the function names in script1.py. The process repeats for script2.py using the same module functions, but with a different calculation for x.

Captions
A programmer writes scripts containing functions and code using those functions. Multiple scripts might define the same functions.
The functions can instead be defined in another file. The file can be imported as a "module".
Playing step 2: The functions can instead be defined in another file. The file can be imported as a "module". Step finished playing

Feedback?

A module's filename should end with ".py"; otherwise, the interpreter will not be able to import the module. The module_name item should match the filename of the module, but without the .py extension. Ex: If a programmer wants to import a module whose filename is HTTPServer.py, the import statement import HTTPServer would be used. Note that import statements are case-sensitive; thus, import ABC is distinct from import aBc.

The interpreter must also be able to find the module to import. The simplest solution is to keep modules in the same directory as the executing script; however, the interpreter can also search the computer's file system for the modules. Later material covers these search mechanisms.

Good practice is to place import statements at the top of a file. There are few useful instances of placing import statements in any location other than the top. The benefit of placing import statements at the top is that a reader of the program can quickly identify the modules required for the program to run. A module being required by another program is often called a dependency.'''

_= Problem("participation activity 11.1.1")
_.create("script1.py", '''import funcs
x = funcs.fct() * funcs.sq()''')
_.create("funcs.py", '''def fct():
    return 42
def sq():
    return 7''')
_.runpy("script1.py", cwd=_.folder)


'11.1 Modules\nModule basics\nThe interactive Python interpreter provides the most basic way to execute Python code. However, all of the defined variables, functions, classes, etc., are lost when a programmer closes the interpreter. Thus, a programmer will typically write Python code in a file, and then pass that file as input to the interpreter. Such a file is called a script.\n\nA programmer may find themselves writing the same function over and over again in multiple scripts, or creating very long and difficult-to-maintain scripts. A solution is to use a module, which is a file containing Python code that can be imported and used by scripts, other modules, or the interactive interpreter. To import a module means to execute the code contained by the module and make the definitions within that module available for use by the importing program.\n\nparticipation activity\n11.1.1: A module is a file containing Python statements and definitions that can be used by other Python sources.\n\

Created file: script1.py with content:
import funcs
x = funcs.fct() * funcs.sq()
Created file: funcs.py with content:
def fct():
    return 42
def sq():
    return 7
Ran command: python script1.py in driver_workspace/participation_activity_11.1.1



In [77]:
'''participation activity
11.1.2: Basic importing of modules.
1)
A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
>>>

Check

Show answer
2)
A file containing Python code that is passed as input to the interpreter is called a _______.

Check

Show answer
3)
A _______ is a file containing Python code that can be imported by a script.

Check

Show answer

Feedback?'''

_ = Problem("participation activity 11.1.2")
_.create("tools.py", '''def greet(name):
    return f"Hello, {name}!"''')
_.runpy("tools.py", cwd=_.folder)
_.create("script.py", '''import tools
print(tools.greet("Alice"))''')
_.runpy("script.py", cwd=_.folder)
_.create("ANSWER.md", '''# Participation Activity 11.1.2
          1) A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
            >>> import tools
            2) A file containing Python code that is passed as input to the interpreter is called a _______.
            Answer: script
            3) A _______ is a file containing Python code that can be imported by a script.
            Answer: module''')

'participation activity\n11.1.2: Basic importing of modules.\n1)\nA programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.\n>>>\n\nCheck\n\nShow answer\n2)\nA file containing Python code that is passed as input to the interpreter is called a _______.\n\nCheck\n\nShow answer\n3)\nA _______ is a file containing Python code that can be imported by a script.\n\nCheck\n\nShow answer\n\nFeedback?'

Created file: tools.py with content:
def greet(name):
    return f"Hello, {name}!"
Ran command: python tools.py in driver_workspace/participation_activity_11.1.2

Created file: script.py with content:
import tools
print(tools.greet("Alice"))
Ran command: python script.py in driver_workspace/participation_activity_11.1.2
Hello, Alice!

Created file: ANSWER.md with content:
# Participation Activity 11.1.2
          1) A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
            >>> import tools
            2) A file containing Python code that is passed as input to the interpreter is called a _______.
            Answer: script
            3) A _______ is a file containing Python code that can be imported by a script.
            Answer: module


In [78]:
'Importing a module\nEvaluating an import statement initiates the following process to load the module:\n\nA check is conducted to determine whether the module has already been imported. If already imported, then the loaded module is used.\nIf not already imported, a new module object is created and inserted in sys.modules.\nThe code in the module is executed in the new module object\'s namespace.\nWhen importing a module, the interpreter first checks to see if that module has already been imported. A dictionary of the loaded modules is stored in sys.modules (available from the sys standard library module). If the module has not yet been loaded, then a new module object is created. A module object is simply a namespace that contains definitions from the module. If the module has already been loaded, then the existing module object is used.\n\nIf a module is not found in sys.modules, then the module is added and the statements within the module\'s code are executed. Definitions in the module\'s code, such as variable assignments and function definitions, are placed in the module\'s namespace. The module is then added to the importing script or module\'s namespace, so the importer can access the definitions. The below animation illustrates.\n\nparticipation activity\n11.1.3: Importing a module.\n\n\n1\n\n2\n\n3\n\n4\n\n5\n\nwebpage has already been imported. Existing module is loaded.\nimport HTTPServer\nimport webpage\n\nmy_ip = HTTPServer.address\n\nwebpage.disp(my_ip)\n\n# ...\nsys.modules\nHTTPServer.py\nwebpage.py\nempty\nHTTPServer\nHTTPServer\nimport webpage\n\naddress = " "\n\n# ...\ndef disp(ip):\n   # ...\n\n# ...\nwebpage\nwebpage\ndisp\nwebpage\naddress\nwebpage\nStatic figure: Begin Python code: import HTTPServer import webpage my_ip = HTTPServer.address webpage.disp(my_ip) # ... End Python code. Begin Python code (HTTPServer.py): import webpage address = " " # ... End Python code. Begin Python code (webpage.py): def disp(ip): # ... # ... End Python code. The sys.modules namespace contains HTTPServer and webpage. The HTTPServer and webpage namespaces are shown with a line drawn to their respective modules in sys.modules. The HTTPServer namespace contains webpage and address. The webpage namespace contains disp. Step 1: sys.modules checks for HTTPServer. A new module object is created. The module is then inserted into sys.modules. sys.modules is initially empty. The main Python code executes the line "import HTTPServer". sys.modules is checked for HTTPServer, a new module is created, and HTTPServer is added to sys.modules. The HTTPServer namespace is initially empty with a line drawn to HTTPServer in sys.modules. Step 2: HTTPServer\'s code is executed in module namespace. sys.modules checks for webpage. The new module object is created and inserted in sys.modules. The HTTPServer code executes the line "import webpage". sys.modules is checked for containing webpage, a new module is created, and webpage is added to sys.modules. The webpage namespace is initially empty with a line drawn to webpage in sys.modules. Step 3: webpage\'s code is executed in module namespace. webpage is added to HTTPServer namespace. webpage\'s code executes, and disp is defined and added to webpage\'s namespace. When webpage\'s code finishes executing, webpage is added to HTTPServer\'s namespace. Step 4: HTTPServer\'s code continues executing. address is defined and added to HTTPServer\'s namespace. Step 5: webpage has already been imported. Existing module is loaded. The main Python code continues and executes the next line "import webpage". sys.modules is checked for webpage, and the existing module is loaded.\n\nCaptions\nsys.modules checks for HTTPServer. A new module object is created. The module is then inserted into sys.modules.\nHTTPServer\'s code is executed in module namespace. sys.modules checks for webpage. The new module object is created and inserted in sys.modules.\nwebpage\'s code is executed in module namespace. webpage is added to HTTPServer namespace.\nHTTPServer\'s code continues executing.\nwebpage has already been imported. Existing module is loaded.\nPlaying step 5: webpage has already been imported. Existing module is loaded. Step finished playing\n\nFeedback?\nExecuting import HTTPServer causes a new module object to be created and added to sys.modules. The code of HTTPServer is executed, which contains another import statement import webpage. Since webpage has not yet been imported, a second new module object is created and added to sys.modules. Execution of the webpage code occurs, which defines a function within the webpage module\'s namespace. Once the webpage module is successfully imported, the execution of HTTPServer\'s code continues, creating new definitions in the HTTPServer module\'s namespace. If the script attempts to import webpage, the already created module object is used.\n\nparticipation activity\n11.1.4: The importing process.\nOrder the events as they occur when the statement import HTTPServer executes, assuming HTTPServer has not been previously imported.\n\nHow to use this tool\nHTTPServer added to importer\'s namespace.\nHTTPServer added to sys.modules.\nModule object created.\nsys.modules checked for HTTPServer.\nHTTPServer code executed.\n1st event\n2nd event\n3rd event\n4th event\n5th event\n\nReset\n\nFeedback?\n'
_ = Problem("participation activity 11.1.4")
_.create("HTTPServer.py", '''import webpage
address = " "# ...''')
_.create("webpage.py", '''def disp(ip):
    print(f"Displaying webpage for IP: {ip}")
# ...''')
_.runpy("HTTPServer.py", cwd=_.folder)
_.create("script.py", '''import HTTPServer
import webpage
my_ip = HTTPServer.address
webpage.disp(my_ip)
# ...''')
_.runpy("script.py", cwd=_.folder)
_.answer('''
         # Participation Activity 11.1.4
         1) sys.modules checked for HTTPServer.
         2) Module object created.
         3) HTTPServer added to sys.modules.
         4) HTTPServer code executed.
         5) HTTPServer added to importer\'s namespace.
         ''')

'Importing a module\nEvaluating an import statement initiates the following process to load the module:\n\nA check is conducted to determine whether the module has already been imported. If already imported, then the loaded module is used.\nIf not already imported, a new module object is created and inserted in sys.modules.\nThe code in the module is executed in the new module object\'s namespace.\nWhen importing a module, the interpreter first checks to see if that module has already been imported. A dictionary of the loaded modules is stored in sys.modules (available from the sys standard library module). If the module has not yet been loaded, then a new module object is created. A module object is simply a namespace that contains definitions from the module. If the module has already been loaded, then the existing module object is used.\n\nIf a module is not found in sys.modules, then the module is added and the statements within the module\'s code are executed. Definitions in the m

Created file: HTTPServer.py with content:
import webpage
address = " "# ...
Created file: webpage.py with content:
def disp(ip):
    print(f"Displaying webpage for IP: {ip}")
# ...
Ran command: python HTTPServer.py in driver_workspace/participation_activity_11.1.4

Created file: script.py with content:
import HTTPServer
import webpage
my_ip = HTTPServer.address
webpage.disp(my_ip)
# ...
Ran command: python script.py in driver_workspace/participation_activity_11.1.4
Displaying webpage for IP:  

Saved answer to ANSWER.md:

         # Participation Activity 11.1.4
         1) sys.modules checked for HTTPServer.
         2) Module object created.
         3) HTTPServer added to sys.modules.
         4) HTTPServer code executed.
         5) HTTPServer added to importer's namespace.
         


In [79]:
'''Using an imported module
Once a module has been imported, the program can access the definitions of a module using attribute reference operations. Ex: my_ip = HTTPServer.address sets my_ip to address defined in HTTPServer.py. The definitions can also be overwritten. Ex: HTTPServer.address = "www.yahoo.com" binds address in HTTPServer to "www.yahoo.com". Note that such changes are temporary and restricted to the current executing Python instance. Ending the program and then re-importing the module would reload the original value of HTTPServer.address.

Consider a file, my_funcs.py, that contains the following:

Figure 11.1.1: Contents of my_funcs.py.
def factorial(num):
    """Calculates and returns the factorial (num!)"""
    x = 1
    for i in range(1, num + 1):
        x *= i

    return x

Feedback?
A programmer can then import my_funcs and use the factorial function as shown below:

Figure 11.1.2: Using factorial from my_funcs.py.
import my_funcs

n = int(input("Enter number: "))
fact = my_funcs.factorial(n)

for i in range(1, n + 1):
    print(i, end=" ")
    if i != n:
        print("*", end=" ")

print(f"= {fact}")
Enter number: 5
1 * 2 * 3 * 4 * 5 = 120
...
Enter number: 3
1 * 2 * 3 = 6

Feedback?
participation activity
11.1.5: Basic usage of imported modules.
Consider a file shapes.py with the following contents:

cr = "#"

def draw_square(size):
    for h in range(size):
        for w in range(size):
            print(cr, end="")
        print() 

def draw_rect(height, width):
    for h in range(height):
        for w in range(width):
            print(cr, end="")
        print()
1)
Complete the import statement to import shapes.py.
import 


Check

Show answer
2)
Complete the statement to call the draw_square function from the shapes module, passing an argument of 3.
shapes.


Check

Show answer
3)
Write a statement that changes the output to use "$" when drawing shapes. (Change the value of shapes.cr.)

Check

Show answer

Feedback?'''
_ = Problem("participation activity 11.1.5")
_.create("shapes.py", 
'''
cr = "#"
def draw_square(size):
    for h in range(size):
        for w in range(size):
            print(cr, end="")
        print()
def draw_rect(height, width):
    for h in range(height):
        for w in range(width):
            print(cr, end="")
        print()''')
_.create("script.py", '''
import shapes
shapes.draw_square(3)
shapes.cr = "$"
shapes.draw_rect(2, 5)''')
_.runpy("script.py", cwd=_.folder)  
_.answer('''
# Participation Activity 11.1.5
1) Complete the import statement to import shapes.py.
import shapes
2) Complete the statement to call the draw_square function from the shapes module, passing an argument of 3.
shapes.draw_square(3)
3) Write a statement that changes the output to use "$" when drawing shapes. (Change the value of shapes.cr.)
shapes.cr = "$"
''')

'Using an imported module\nOnce a module has been imported, the program can access the definitions of a module using attribute reference operations. Ex: my_ip = HTTPServer.address sets my_ip to address defined in HTTPServer.py. The definitions can also be overwritten. Ex: HTTPServer.address = "www.yahoo.com" binds address in HTTPServer to "www.yahoo.com". Note that such changes are temporary and restricted to the current executing Python instance. Ending the program and then re-importing the module would reload the original value of HTTPServer.address.\n\nConsider a file, my_funcs.py, that contains the following:\n\nFigure 11.1.1: Contents of my_funcs.py.\ndef factorial(num):\n    """Calculates and returns the factorial (num!)"""\n    x = 1\n    for i in range(1, num + 1):\n        x *= i\n\n    return x\n\nFeedback?\nA programmer can then import my_funcs and use the factorial function as shown below:\n\nFigure 11.1.2: Using factorial from my_funcs.py.\nimport my_funcs\n\nn = int(input

Created file: shapes.py with content:

cr = "#"
def draw_square(size):
    for h in range(size):
        for w in range(size):
            print(cr, end="")
        print()
def draw_rect(height, width):
    for h in range(height):
        for w in range(width):
            print(cr, end="")
        print()
Created file: script.py with content:

import shapes
shapes.draw_square(3)
shapes.cr = "$"
shapes.draw_rect(2, 5)
Ran command: python script.py in driver_workspace/participation_activity_11.1.5
###
###
###
$$$$$
$$$$$

Saved answer to ANSWER.md:

# Participation Activity 11.1.5
1) Complete the import statement to import shapes.py.
import shapes
2) Complete the statement to call the draw_square function from the shapes module, passing an argument of 3.
shapes.draw_square(3)
3) Write a statement that changes the output to use "$" when drawing shapes. (Change the value of shapes.cr.)
shapes.cr = "$"



In [80]:
'''challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
arithmetic.py
import arithmetic

def calculate(number):
    return number - 5

print(arithmetic.calculate(3))
print(calculate(3))
1
2
3
4
Check
Next
1
2
3
4

Feedback?

challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
arithmetic.py
def calculate(number):
    return number * 5
8
-2

1
2
3
4'''

_ = Problem("challenge activity 11.1.1 part 1")

_.create("arithmetic.py", '''
def calculate(number):
    return number * 5''')

_.create("main.py", '''
import arithmetic
def calculate(number):
    return number - 5
print(arithmetic.calculate(3))
print(calculate(3))''')

_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 1
1) 15
2) -2
''')


"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\narithmetic.py\nimport arithmetic\n\ndef calculate(number):\n    return number - 5\n\nprint(arithmetic.calculate(3))\nprint(calculate(3))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\n\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\narithmetic.py\ndef calculate(number):\n    return number * 5\n8\n-2\n\n1\n2\n3\n4"

Created file: arithmetic.py with content:

def calculate(number):
    return number * 5
Created file: main.py with content:

import arithmetic
def calculate(number):
    return number - 5
print(arithmetic.calculate(3))
print(calculate(3))
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_1
15
-2

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 1
1) 15
2) -2



In [81]:
'''challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
data.py
arithmetic.py
import arithmetic
import data

def calculate(number):
    return number * 4

print(calculate(data.medium))
print(arithmetic.calculate(data.medium))
1
2
3
4
Check
Next
1
2
3
4

Feedback?challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
data.py
arithmetic.py
small = 9
medium = 80
large = 300
1
2
3
4
Check
Next
1
2
3
4

Feedback?challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
data.py
arithmetic.py
def calculate(number):
    return number + 2
1
2
3
4
Check
Next
1
2
3
4

Feedback?'''
_ = Problem("challenge activity 11.1.1 part 2")
_.create("data.py", '''
small = 9
medium = 80
large = 300''')
_.create("arithmetic.py", '''
def calculate(number):
    return number + 2''')
_.create("main.py", '''
import arithmetic
import data
def calculate(number):
    return number * 4
print(calculate(data.medium))
print(arithmetic.calculate(data.medium))''')
_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 2
1) 320
2) 82
''')

"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\ndata.py\narithmetic.py\nimport arithmetic\nimport data\n\ndef calculate(number):\n    return number * 4\n\nprint(calculate(data.medium))\nprint(arithmetic.calculate(data.medium))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\ndata.py\narithmetic.py\nsmall = 9\nmedium = 80\nlarge = 300\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\ndata.py\narithmetic.py\ndef calculate(number):\n    return number + 2\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?"

Created file: data.py with content:

small = 9
medium = 80
large = 300
Created file: arithmetic.py with content:

def calculate(number):
    return number + 2
Created file: main.py with content:

import arithmetic
import data
def calculate(number):
    return number * 4
print(calculate(data.medium))
print(arithmetic.calculate(data.medium))
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_2
320
82

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 2
1) 320
2) 82



In [82]:
'challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\nimport green\nimport red\n\nprint(green.dull)\nprint(red.dull)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "viridian"\nbright = "forest"\nmedium = "chartreuse"\ndull = "olive"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "vermilion"\nbright = "crimson"\nmedium = "salmon"\ndull = "coral"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?'
_ = Problem("challenge activity 11.1.1 part 3")
_.create("green.py", '''
dark = "viridian"
bright = "forest"
medium = "chartreuse"
dull = "olive"''')
_.create("red.py", '''
dark = "vermilion"
bright = "crimson"
medium = "salmon"
dull = "coral"''')
_.create("main.py", '''
import green
import red
print(green.dull)
print(red.dull)''')
_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 3
1) olive
2) coral
''')

'challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\nimport green\nimport red\n\nprint(green.dull)\nprint(red.dull)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "viridian"\nbright = "forest"\nmedium = "chartreuse"\ndull = "olive"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "vermilion"\nbright = "crimson"\nmedium = "salmon"\ndull = "coral"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?'

Created file: green.py with content:

dark = "viridian"
bright = "forest"
medium = "chartreuse"
dull = "olive"
Created file: red.py with content:

dark = "vermilion"
bright = "crimson"
medium = "salmon"
dull = "coral"
Created file: main.py with content:

import green
import red
print(green.dull)
print(red.dull)
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_3
olive
coral

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 3
1) olive
2) coral



In [83]:
"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport first\n\ndef fct_a(number):\n    return number ** 2\n\ndef fct_b(number):\n    return number * 1\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\nprint(fct_c(7))\nprint(first.fct_c(7))\nprint(first.fct_d(7))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport second\n\ndef fct_a(number):\n    return number + 9\n\ndef fct_b(number):\n    return number * 6\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\ndef fct_d(number):\n    return second.fct_c(number)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\ndef fct_a(number):\n    return number - 3\n\ndef fct_b(number):\n    return number + 8\n\ndef fct_c(number):\n    return number * 2\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?"
_ = Problem("challenge activity 11.1.1 part 4")
_.create("first.py", '''
import second

def fct_a(number):
    return number + 3

def fct_b(number):
    return number * 8

def fct_c(number):
    return fct_a(number) - fct_b(number)

def fct_d(number):
    return second.fct_c(number)''')
_.create("second.py", '''
def fct_a(number):
    return number - 6

def fct_b(number):
    return number + 1

def fct_c(number):
    return number * 7''')
_.create("main.py", '''
import first

def fct_a(number):
    return number ** 2

def fct_b(number):
    return number * 5

def fct_c(number):
    return fct_a(number) - fct_b(number)

print(fct_c(4))
print(first.fct_c(4))
print(first.fct_d(4))''')
_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 4
1) 0
2) -25
3) -28
''')

"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport first\n\ndef fct_a(number):\n    return number ** 2\n\ndef fct_b(number):\n    return number * 1\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\nprint(fct_c(7))\nprint(first.fct_c(7))\nprint(first.fct_d(7))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport second\n\ndef fct_a(number):\n    return number + 9\n\ndef fct_b(number):\n    return number * 6\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\ndef fct_d(number):\n    return second.fct_c(number)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nm

Created file: first.py with content:

import second

def fct_a(number):
    return number + 3

def fct_b(number):
    return number * 8

def fct_c(number):
    return fct_a(number) - fct_b(number)

def fct_d(number):
    return second.fct_c(number)
Created file: second.py with content:

def fct_a(number):
    return number - 6

def fct_b(number):
    return number + 1

def fct_c(number):
    return number * 7
Created file: main.py with content:

import first

def fct_a(number):
    return number ** 2

def fct_b(number):
    return number * 5

def fct_c(number):
    return fct_a(number) - fct_b(number)

print(fct_c(4))
print(first.fct_c(4))
print(first.fct_d(4))
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_4
-4
-25
28

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 4
1) 0
2) -25
3) -28



In [91]:
'''challenge activity
11.1.2: Modules.
712910.5105864.qx3zqy7

Jump to level 1
Integers base_in_cm and height_in_cm are read from input, representing a triangle's base and height in centimeters. Module si_units defines cm_to_mm() and module tri_shape defines area(). Import modules si_units and tri_shape into main.py.

Click here for example
Ex: If the input is:
8
3
then the output is:

8 cm is 80 mm
3 cm is 30 mm
Triangle area in mm^2: 1200.0
'''
_q = Problem("challenge activity 11.1.2 part 1")
_q.create("si_units.py", '''
def cm_to_mm(cm):
    return cm * 10''')
_q.create("tri_shape.py", '''
def area(base, height):
    return base * height / 2''')
_q.create("main.py", '''
import si_units
import tri_shape
base_in_cm = int(input())
height_in_cm = int(input())

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

print(f"{base_in_cm} cm is {base_in_mm} mm")
print(f"{height_in_cm} cm is {height_in_mm} mm")
print(f"Triangle area in mm^2: {tri_shape.area(base_in_mm, height_in_mm)}")''')

_q.runpy("main.py", inputs=["8", "3"])
_q.answer('''
# Challenge Activity 11.1.2 Part 1
1) import si_units
import tri_shape
2) 8 cm is 80 mm
3) 3 cm is 30 mm
4) Triangle area in mm^2: 1200.0
''')


"challenge activity\n11.1.2: Modules.\n712910.5105864.qx3zqy7\n\nJump to level 1\nIntegers base_in_cm and height_in_cm are read from input, representing a triangle's base and height in centimeters. Module si_units defines cm_to_mm() and module tri_shape defines area(). Import modules si_units and tri_shape into main.py.\n\nClick here for example\nEx: If the input is:\n8\n3\nthen the output is:\n\n8 cm is 80 mm\n3 cm is 30 mm\nTriangle area in mm^2: 1200.0\n"

Created file: si_units.py with content:

def cm_to_mm(cm):
    return cm * 10
Created file: tri_shape.py with content:

def area(base, height):
    return base * height / 2
Created file: main.py with content:

import si_units
import tri_shape
base_in_cm = int(input())
height_in_cm = int(input())

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

print(f"{base_in_cm} cm is {base_in_mm} mm")
print(f"{height_in_cm} cm is {height_in_mm} mm")
print(f"Triangle area in mm^2: {tri_shape.area(base_in_mm, height_in_mm)}")
Ran command: python main.py in driver_workspace/challenge_activity_11.1.2_part_1
8 cm is 80 mm
3 cm is 30 mm
Triangle area in mm^2: 1200.0

Saved answer to ANSWER.md:

# Challenge Activity 11.1.2 Part 1
1) import si_units
import tri_shape
2) 8 cm is 80 mm
3) 3 cm is 30 mm
4) Triangle area in mm^2: 1200.0



In [95]:
'''challenge activity
11.1.2: Modules.
712910.5105864.qx3zqy7

Jump to level 1
Integers base_in_cm and height_in_cm are read from input, representing a triangle's base and height in centimeters. si_units's cm_to_mm() converts a measurement from centimeters to millimeters.

Assign area_cm2 with the value returned by tri_shape's area() called with base_in_cm and height_in_cm as the arguments, respectively.
Find the triangle's base and height in millimeters. Then, assign area_mm2 with the value returned by tri_shape's area() called with the triangle's base and height in millimeters as the arguments, respectively.
Click here for example
Ex: If the input is:
7
8
then the output is:

Triangle area is 28.0 cm^2 or 2800.0 mm^2.


main.py
import si_units
import tri_shape

base_in_cm = int(input())
height_in_cm = int(input())

""" Your code goes here """

print(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")
si_units.py
def cm_to_mm(cm):
	return cm * 10

tri_shape.py
def area(base, height):
	return base * height / 2
'''
_ = Problem("challenge activity 11.1.2 part 2")
_.create("si_units.py", '''
def cm_to_mm(cm):
    return cm * 10''')
_.create("tri_shape.py", '''
def area(base, height):
    return base * height / 2''')
_.create("main.py", '''
import si_units
import tri_shape

base_in_cm = int(input())
height_in_cm = int(input())

area_cm2 = tri_shape.area(base_in_cm, height_in_cm)

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

area_mm2 = tri_shape.area(base_in_mm, height_in_mm)

print(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")''')
_.runpy("main.py", inputs=["7", "8"])
_.answer('''
# Challenge Activity 11.1.2 Part 2
1) area_cm2 = tri_shape.area(base_in_cm, height_in_cm)
2) base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)
area_mm2 = tri_shape.area(base_in_mm, height_in_mm)
3) Triangle area is 28.0 cm^2 or 2800.0 mm^2.
''')

'challenge activity\n11.1.2: Modules.\n712910.5105864.qx3zqy7\n\nJump to level 1\nIntegers base_in_cm and height_in_cm are read from input, representing a triangle\'s base and height in centimeters. si_units\'s cm_to_mm() converts a measurement from centimeters to millimeters.\n\nAssign area_cm2 with the value returned by tri_shape\'s area() called with base_in_cm and height_in_cm as the arguments, respectively.\nFind the triangle\'s base and height in millimeters. Then, assign area_mm2 with the value returned by tri_shape\'s area() called with the triangle\'s base and height in millimeters as the arguments, respectively.\nClick here for example\nEx: If the input is:\n7\n8\nthen the output is:\n\nTriangle area is 28.0 cm^2 or 2800.0 mm^2.\n\n\nmain.py\nimport si_units\nimport tri_shape\n\nbase_in_cm = int(input())\nheight_in_cm = int(input())\n\n""" Your code goes here """\n\nprint(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")\nsi_units.py\ndef cm_to_mm(cm):\n\treturn cm * 1

Created file: si_units.py with content:

def cm_to_mm(cm):
    return cm * 10
Created file: tri_shape.py with content:

def area(base, height):
    return base * height / 2
Created file: main.py with content:

import si_units
import tri_shape

base_in_cm = int(input())
height_in_cm = int(input())

area_cm2 = tri_shape.area(base_in_cm, height_in_cm)

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

area_mm2 = tri_shape.area(base_in_mm, height_in_mm)

print(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")
Ran command: python main.py in driver_workspace/challenge_activity_11.1.2_part_2
Triangle area is 28.0 cm^2 or 2800.0 mm^2.

Saved answer to ANSWER.md:

# Challenge Activity 11.1.2 Part 2
1) area_cm2 = tri_shape.area(base_in_cm, height_in_cm)
2) base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)
area_mm2 = tri_shape.area(base_in_mm, height_in_mm)
3) Triangle area is 28.0 cm^2 or 2800.0 mm^2.



In [98]:
'''challenge activity
11.1.2: Modules.
712910.5105864.qx3zqy7

Jump to level 1
Perform the following tasks:

Assign has_color with the value returned by search's find() called with one_sentence and colors's search_list as the arguments, respectively.
Assign has_person_and_clothing with the value returned by combining the following, using logical AND:
search's find() called with one_sentence and persons's search_list as the arguments, respectively.
search's find() called with one_sentence and clothing's search_list as the arguments, respectively.
Click here for example
Ex: If the input is A pink t-shirt was sent to Del., then the output is:

The sentence mentions a color.
The sentence mentions a person and a piece of clothing.


main.py
import persons
import colors
import clothing
import search

one_sentence = input()

""" Your code goes here """

if has_color:
	print("The sentence mentions a color.")

if has_person_and_clothing:
	print("The sentence mentions a person and a piece of clothing.")
persons.py
search_list = ["Del", "Noa", "Ada"]
colors.py
search_list = ["red", "white", "pink"]
clothing.py
search_list = ["shirt", "jacket", "t-shirt"]
search.py
def find(one_sentence, search_list):
    words = one_sentence[:-1].split()
    for word in words:
        if word in search_list:
            return True
    return False
1

2

3

Check

Next level
1
2
3

Feedback?'''
_ = Problem("challenge activity 11.1.2 part 3")
_.create("persons.py", '''
search_list = ["Del", "Noa", "Ada"]''')
_.create("colors.py", '''
search_list = ["red", "white", "pink"]''')
_.create("clothing.py", '''
search_list = ["shirt", "jacket", "t-shirt"]''')
_.create("search.py", '''
def find(one_sentence, search_list):
    words = one_sentence[:-1].split()
    for word in words:
        if word in search_list:
            return True
    return False''')
_.create("main.py", '''
import persons
import colors
import clothing
import search
one_sentence = input()
has_color = search.find(one_sentence, colors.search_list)
has_person_and_clothing = search.find(one_sentence, persons.search_list) and search.find(one_sentence, clothing.search_list)
if has_color:
    print("The sentence mentions a color.") 
if has_person_and_clothing:
    print("The sentence mentions a person and a piece of clothing.")''')
_.runpy("main.py", inputs=["A pink t-shirt was sent to Del."])
_.answer('''
# Challenge Activity 11.1.2 Part 3
1) has_color = search.find(one_sentence, colors.search_list)
2) has_person_and_clothing = search.find(one_sentence, persons.search_list) and search.find(one_sentence, clothing.search_list)
3) The sentence mentions a color.
The sentence mentions a person and a piece of clothing.
''')

'challenge activity\n11.1.2: Modules.\n712910.5105864.qx3zqy7\n\nJump to level 1\nPerform the following tasks:\n\nAssign has_color with the value returned by search\'s find() called with one_sentence and colors\'s search_list as the arguments, respectively.\nAssign has_person_and_clothing with the value returned by combining the following, using logical AND:\nsearch\'s find() called with one_sentence and persons\'s search_list as the arguments, respectively.\nsearch\'s find() called with one_sentence and clothing\'s search_list as the arguments, respectively.\nClick here for example\nEx: If the input is A pink t-shirt was sent to Del., then the output is:\n\nThe sentence mentions a color.\nThe sentence mentions a person and a piece of clothing.\n\n\nmain.py\nimport persons\nimport colors\nimport clothing\nimport search\n\none_sentence = input()\n\n""" Your code goes here """\n\nif has_color:\n\tprint("The sentence mentions a color.")\n\nif has_person_and_clothing:\n\tprint("The sentenc

Created file: persons.py with content:

search_list = ["Del", "Noa", "Ada"]
Created file: colors.py with content:

search_list = ["red", "white", "pink"]
Created file: clothing.py with content:

search_list = ["shirt", "jacket", "t-shirt"]
Created file: search.py with content:

def find(one_sentence, search_list):
    words = one_sentence[:-1].split()
    for word in words:
        if word in search_list:
            return True
    return False
Created file: main.py with content:

import persons
import colors
import clothing
import search
one_sentence = input()
has_color = search.find(one_sentence, colors.search_list)
has_person_and_clothing = search.find(one_sentence, persons.search_list) and search.find(one_sentence, clothing.search_list)
if has_color:
    print("The sentence mentions a color.") 
if has_person_and_clothing:
    print("The sentence mentions a person and a piece of clothing.")
Ran command: python main.py in driver_workspace/challenge_activity_11.1.2_part_3
The sentenc

In [106]:
class scratchwork:
    _=Problem("scratchwork")

    _.create("test.py", '''def fct():
        return "Hello from test.py!"''')
    _.runpy("test.py", cwd=_.folder)

    print(_.get_file("test.py"))

Created file: test.py with content:
def fct():
        return "Hello from test.py!"
Ran command: python test.py in driver_workspace/scratchwork

def fct():
        return "Hello from test.py!"


11.2 Finding modules

https://learn.zybooks.com/zybook/CPPCS2520NguyenSpring2026/chapter/11/section/2

In [109]:
'''11.2 Finding modules
Importing a module begins a search to find the corresponding file on the computer's file system. The interpreter first checks for a matching built-in module. A built-in module comes pre-installed with Python; examples of built-in modules include sys, time, and math. If no matching built-in module is found, then the interpreter searches for the necessary module in sys.path, a built-in Python variable located in the sys module that has a list of directories containing modules. A programmer must be careful to not give a name to a module that is already used by a built-in module. In such cases, the interpreter would load the built-in module because built-in names are checked first.

The sys.path variable initially contains the following directories:


The directory of the executing script.
A list of directories specified by the environment variable PYTHONPATH.
The directory where Python is installed.
For simple programs, a module might be placed in the same directory. Larger projects might contain tens or hundreds of modules or use third-party modules located in different directories. In such cases, a programmer might set the environment variable PYTHONPATH in the operating system. An operating system environment variable is much like a variable in a Python script, except that an environment variable is stored by the computer's operating system and can be accessed by every program running on the computer. In Windows, a user can set the value of PYTHONPATH permanently through the control panel, or temporarily on a single instance of a command terminal (cmd.exe) using the command set PYTHONPATH="c:\dir1;c:\other\directory".

activity
11.2.1: Finding modules.
1)
When an import statement executes, the interpreter immediately checks the current directory for a matching file.
2)
The environment variable PYTHONPATH can be set to specify optional directories where modules are located.
3)
math.py is a good name for a new module.

Feedback?
How was this section?

|


Provide section feedback'''
_ = Problem("activity 11.2.1")
_.create("script.py", '''
import math
print(math.sqrt(16))
''')
_.runpy("script.py", cwd=_.folder)
_.answer('''
# Activity 11.2.1
1) False
2) True
3) False
''')


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_69779/1390622080.py:1: SyntaxWarning: invalid escape sequence '\d'
  '''11.2 Finding modules


'11.2 Finding modules\nImporting a module begins a search to find the corresponding file on the computer\'s file system. The interpreter first checks for a matching built-in module. A built-in module comes pre-installed with Python; examples of built-in modules include sys, time, and math. If no matching built-in module is found, then the interpreter searches for the necessary module in sys.path, a built-in Python variable located in the sys module that has a list of directories containing modules. A programmer must be careful to not give a name to a module that is already used by a built-in module. In such cases, the interpreter would load the built-in module because built-in names are checked first.\n\nThe sys.path variable initially contains the following directories:\n\n\nThe directory of the executing script.\nA list of directories specified by the environment variable PYTHONPATH.\nThe directory where Python is installed.\nFor simple programs, a module might be placed in the same 

Created file: script.py with content:

import math
print(math.sqrt(16))

Ran command: python script.py in driver_workspace/activity_11.2.1
4.0

Saved answer to ANSWER.md:

# Activity 11.2.1
1) False
2) True
3) False



In [117]:
#builtinmodule
_= Problem("scratchwork")
_.create("math.py", '''def sqrt(x):
    return "This is not the real math.sqrt!"''')
_.runpy("math.py", cwd=_.folder)
_.create("script.py", '''import math
print(math.sqrt(16))''')
_.runpy("script.py", cwd=_.folder)
#The output is 4.0, which is the result of the built-in math.sqrt function, not the sqrt function defined in math.py. This is because the interpreter checks for built-in modules before checking for files in the current directory. Since math is a built-in module, it is loaded instead of the math.py file in the current directory.
#PYTHONPATH
_.create("module_in_dir.py", '''def greet():
    return "Hello from module_in_dir!"''')
_.runpy("module_in_dir.py", cwd=_.folder)
import os
import sys
import importlib
# Add the current directory to PYTHONPATH
os.environ["PYTHONPATH"] = os.getcwd()
# Now we can import module_in_dir
if str(_.folder) not in sys.path:
    sys.path.insert(0, str(_.folder))

module_in_dir = importlib.import_module("module_in_dir")

print(module_in_dir.greet())  

Created file: math.py with content:
def sqrt(x):
    return "This is not the real math.sqrt!"
Ran command: python math.py in driver_workspace/scratchwork

Created file: script.py with content:
import math
print(math.sqrt(16))
Ran command: python script.py in driver_workspace/scratchwork
This is not the real math.sqrt!

Created file: module_in_dir.py with content:
def greet():
    return "Hello from module_in_dir!"
Ran command: python module_in_dir.py in driver_workspace/scratchwork

Hello from module_in_dir!


11.3 Importing specific names from a module

https://learn.zybooks.com/zybook/CPPCS2520NguyenSpring2026/chapter/11/section/3

In [ ]:
'''11.3 Importing specific names from a module
A programmer can specify names to import from a module by using the from keyword in an import statement:

Construct 11.3.1: Importing specific names from a module.
from module_name import name1, name2, ...

Feedback?
A normal import statement, such as import HTTPServer, adds the new module into the global namespace, after which a programmer can access names through attribute reference operations (e.g., HTTPServer.address). In contrast, using from adds only the specified names. A statement such as from HTTPServer import address copies only the address variable from HTTPServer into the importing module's namespace. The following animation illustrates.

participation activity
11.3.1: 'import x' vs.'from x import y'.


1

2

3

from my_mod import calc only copies calc from the my_mod namespace into the global namespace.
import my_mod

y = 100
z = my_mod.calc(y)
x = 55


def calc(a):
   # ...
global namespacemy_mod namespace
x
calc
my_mod
y
z
from my_mod import calc

y = 100
z = calc(y)
global namespace
calc
y
z
my_mod.py
Static figure: Begin Python code (my_mod.py): x = 55 def calc(a): # ... End Python code. Begin Python code 1: import my_mod y = 100 z = my_mod.calc(y) End Python code. Python code 1's global namespace contains my_mod, y, and z. my_mod references the my_mod namespace, which contains x and calc. Begin Python code 2: from my_mod import calc y = 100 z = calc(y) End Python code. Python code 2's global namespace contains calc, y, and z. Step 1: import my_mod adds my_mod into the global namespace. The global namespace is initially empty, and executing the line "import my_mod" adds my_mod to the global namespace. The code in my_mod is executed and the my_mod namespace contains x and calc. Step 2: calc can be accessed using attribute reference operations. The line "y = 100" is executed and y is added to the global namespace. The line "z = my_mod.calc(y)" is executed and the value of calc is accessed through the my_mod reference in the global namespace. z is added to the global namespace. Step 3: from my_mod import calc only copies calc from the my_mod namespace into the global namespace. The second Python program executes the line "from my_mod import calc" with the second global namespace initially empty. calc is copied from the my_mod namespace to the global namespace. The program continues to execute and adds y to the global namespace. The line "z = calc(y)" is executed and calc is accessed from the global namespace. z is added to the global namespace.

Captions
import my_mod adds my_mod into the global namespace.
calc can be accessed using attribute reference operations.
from my_mod import calc only copies calc from the my_mod namespace into the global namespace.
Playing step 3: from my_mod import calc only copies calc from the my_mod namespace into the global namespace. Step finished playing

Feedback?
Using "from" changes how an imported name is used in a program.

Table 11.3.1: 'import module' vs. 'from module import names'.
Description	Example import statement	Using imported names
Import an entire module	
import HTTPServer
print(HTTPServer.address)
Import specific name from a module	
from HTTPServer import address
print(address)

Feedback?
The program below imports names from the hashlib module, a Python standard library module that contains a number of algorithms for creating a secure hash of a text message. A secure hash correlates exactly to a single series of characters. A sender of an email might create and send a secure hash along with the contents of the message. The email's recipient creates their own secure hash from the message contents and compares it to the received hash to detect if the message was changed.

Figure 11.3.1: Using the from keyword to import specific names.
from hashlib import md5, sha1

text = input('Enter text to hash ("q" to quit): ')

while text != "q":
    algorithm = input("Enter algorithm (md5/sha1): ")
    if algorithm == "md5":
        output = md5(text.encode("utf-8"))
    elif algorithm == "sha1":
        output = sha1(text.encode("utf-8"))
    else:
        output = "Invalid algorithm selection"
    print(f"Hash value: {output.hexdigest()}")

    text = input('\nEnter text to hash ("q" to quit): ')
Enter text to hash ("q" to quit): Whether 'tis nobler in the mind to suffer...
Enter algorithm (md5/sha1): md5
Hash value: 5b39e6686305363a2d60a4162fe3d012

Enter text to hash ("q" to quit): ...the slings and arrows of outrageous fortune,...
Enter algorithm (md5/sha1): sha1
Hash value: 70c137974ad24691c1bb6cf8114aa2e3172ef910

Enter text to hash ("q" to quit): q
The hashlib library requires argument strings to md5 and sha1 be encoded. Above, we encode the text using UTF-8 before passing to one of the hashing algorithms.


Feedback?
Try 11.3.1: Extending the hash example.

Full screen
Improve the hashing example from above by adding a new algorithm. Import the sha224() function from hashlib, and extend the user interface to allow that function to be called.
```# FIXME: Import sha224 also
from hashlib import md5, sha1

text = input("Enter text to hash (\"q\" to quit): ")

# FIXME: Add sha224 to prompt
algorithm = input("\nEnter algorithm (md5/sha1): ")
if algorithm == "md5":
    output = md5(text.encode("utf-8"))
elif algorithm == "sha1":
    output = sha1(text.encode("utf-8"))
    # FIXME: Add check for sha224
else:
    output = "Invalid algorithm selection"

print(f"\nHash value: {output.hexdigest()}")```
'''
_ = Problem("Try 11.3.1")
_.create("main.py", r'''
from hashlib import md5, sha1, sha224
text = input("Enter text to hash (\"q\" to quit): ")
algorithm = input("\nEnter algorithm (md5/sha1/sha224): ")
if algorithm == "md5":
    output = md5(text.encode("utf-8"))
elif algorithm == "sha1":
    output = sha1(text.encode("utf-8"))
elif algorithm == "sha224":
    output = sha224(text.encode("utf-8"))
else:
    output = "Invalid algorithm selection"
print(f"\nHash value: {output.hexdigest()}")''')
_.runpy("main.py", inputs=["Hello, world!", "sha224"])
_.answer('''# Try 11.3.1
1) from hashlib import md5, sha1, sha224
2) Enter algorithm (md5/sha1/sha224):
''')

'11.3 Importing specific names from a module\nA programmer can specify names to import from a module by using the from keyword in an import statement:\n\nConstruct 11.3.1: Importing specific names from a module.\nfrom module_name import name1, name2, ...\n\nFeedback?\nA normal import statement, such as import HTTPServer, adds the new module into the global namespace, after which a programmer can access names through attribute reference operations (e.g., HTTPServer.address). In contrast, using from adds only the specified names. A statement such as from HTTPServer import address copies only the address variable from HTTPServer into the importing module\'s namespace. The following animation illustrates.\n\nparticipation activity\n11.3.1: \'import x\' vs.\'from x import y\'.\n\n\n1\n\n2\n\n3\n\nfrom my_mod import calc only copies calc from the my_mod namespace into the global namespace.\nimport my_mod\n\ny = 100\nz = my_mod.calc(y)\nx = 55\n\n\ndef calc(a):\n   # ...\nglobal namespacemy_m

Created file: main.py with content:


from hashlib import md5, sha1, sha224
text = input("Enter text to hash (\"q\" to quit): ")
algorithm = input("\nEnter algorithm (md5/sha1/sha224): ")
if algorithm == "md5":
    output = md5(text.encode("utf-8"))
elif algorithm == "sha1":
    output = sha1(text.encode("utf-8"))
elif algorithm == "sha224":
    output = sha224(text.encode("utf-8"))
else:
    output = "Invalid algorithm selection"
print(f"\nHash value: {output.hexdigest()}")
Ran command: python main.py in driver_workspace/try_11.3.1
Enter text to hash ("q" to quit): 
Enter algorithm (md5/sha1/sha224): 
Hash value: 8552d8b7a7dc5476cb9e25dee69a8091290764b7f2a64fe6e78e9568

Saved answer to ANSWER.md:
# Try 11.3.1
1) from hashlib import md5, sha1, sha224
2) Enter algorithm (md5/sha1/sha224):



In [ ]:
'''All names from a module can be imported directly by using a "*" character, as in the statement from HTTPServer import *. A common error is to use the import * syntax in modules and scripts, which makes identification of dependencies and the origins of variables difficult for a reader of the code to understand. Good practice is to limit the use of import * syntax to interactive interpreter sessions.

participation activity
11.3.2: Importing specific names.
my_funcs.py contains definitions for the factorial() and squared() functions.

1)
Write a statement that imports only the function factorial from my_funcs.py.

Check

Show answer
2)
The following code uses functions defined in my_funcs.py. Complete the import statement at the top of the program.


a = 5

print(f"a! = {my_funcs.factorial(a)}")

print(f"a^2 = {my_funcs.squared(a)}")

 

Check

Show answer
3)
The following code uses functions defined in my_funcs.py. Complete the import statement at the top of the program.


a = 5

print(f"a! = {factorial(a)}")

print(f"a^2 = {squared(a)}")

 

Check

Show answer

Feedback?'''
_ = Problem("participation activity 11.3.2")
_.create("my_funcs.py", '''def factorial(num):
    """Calculates and returns the factorial (num!)"""
    x = 1
    for i in range(1, num + 1):
        x *= i

    return x

def squared(num):
    """Calculates and returns the square of a number"""
    return num * num''')
_.create("script1.py", '''import my_funcs
a = 5
print(f"a! = {my_funcs.factorial(a)}")
print(f"a^2 = {my_funcs.squared(a)}")
''')
_.runpy("script1.py", cwd=_.folder)
_.create("script2.py", '''from my_funcs import factorial, squared
a = 5
print(f"a! = {factorial(a)}")
print(f"a^2 = {squared(a)}")''')
_.runpy("script2.py", cwd=_.folder)
_.answer('''# Participation Activity 11.3.2
1) from my_funcs import factorial
2) import my_funcs
3) from my_funcs import factorial, squared
''')

'All names from a module can be imported directly by using a "*" character, as in the statement from HTTPServer import *. A common error is to use the import * syntax in modules and scripts, which makes identification of dependencies and the origins of variables difficult for a reader of the code to understand. Good practice is to limit the use of import * syntax to interactive interpreter sessions.\n\nparticipation activity\n11.3.2: Importing specific names.\nmy_funcs.py contains definitions for the factorial() and squared() functions.\n\n1)\nWrite a statement that imports only the function factorial from my_funcs.py.\n\nCheck\n\nShow answer\n2)\nThe following code uses functions defined in my_funcs.py. Complete the import statement at the top of the program.\n\n\na = 5\n\nprint(f"a! = {my_funcs.factorial(a)}")\n\nprint(f"a^2 = {my_funcs.squared(a)}")\n\n\n\nCheck\n\nShow answer\n3)\nThe following code uses functions defined in my_funcs.py. Complete the import statement at the top of 

Created file: my_funcs.py with content:
def factorial(num):
    """Calculates and returns the factorial (num!)"""
    x = 1
    for i in range(1, num + 1):
        x *= i

    return x

def squared(num):
    """Calculates and returns the square of a number"""
    return num * num
Created file: script1.py with content:
import my_funcs
a = 5
print(f"a! = {my_funcs.factorial(a)}")
print(f"a^2 = {my_funcs.squared(a)}")

Ran command: python script1.py in driver_workspace/participation_activity_11.3.2
a! = 120
a^2 = 25

Created file: script2.py with content:
from my_funcs import factorial, squared
a = 5
print(f"a! = {factorial(a)}")
print(f"a^2 = {squared(a)}")
Ran command: python script2.py in driver_workspace/participation_activity_11.3.2
a! = 120
a^2 = 25

Saved answer to ANSWER.md:
# Participation Activity 11.3.2
1) from my_funcs import factorial
2) from my_funcs import factorial, squared
3) from my_funcs import factorial, squared



In [ ]:
'''challenge activity
11.3.1: Importing names directly.
712910.5105864.qx3zqy7

Jump to level 1
Module pie_shop defines buy_apple_pie() and module apple_pie defines bake(). Import the names buy_apple_pie and bake into main.py.

Click here for example
Ex: If the input is:
Eve
10
then the output is:

Eve buys a pie with 10 apples.
10 apples are baked into one apple pie.


main.py

""" Your code goes here """

customer_name = input()
num_apples = int(input())

buy_apple_pie(customer_name, num_apples)
bake(num_apples)
pie_shop.py
def buy_apple_pie(customer_name, num_apples):
	print(f"{customer_name} buys a pie with {num_apples} apples.")

apple_pie.py
def bake(num_apples):
	print(f"{num_apples} apples are baked into one apple pie.")

1

2

3

Check

Next level
1
2
3'''
_ = Problem("challenge activity 11.3.1 part 1")
_.create("pie_shop.py", '''def buy_apple_pie(customer_name, num_apples):
    print(f"{customer_name} buys a pie with {num_apples} apples.")''')
_.create("apple_pie.py", '''def bake(num_apples):
    print(f"{num_apples} apples are baked into one apple pie.")''')
_.create("main.py", '''from pie_shop import buy_apple_pie
from apple_pie import bake
customer_name = input()
num_apples = int(input())
buy_apple_pie(customer_name, num_apples)
bake(num_apples)''')
_.runpy("main.py", inputs=["Eve", "10"])
_.answer('''# Challenge Activity 11.3.1
1) from pie_shop import buy_apple_pie
from apple_pie import bake
2) Eve buys a pie with 10 apples.
3) 10 apples are baked into one apple pie.
''')

'challenge activity\n11.3.1: Importing names directly.\n712910.5105864.qx3zqy7\n\nJump to level 1\nModule pie_shop defines buy_apple_pie() and module apple_pie defines bake(). Import the names buy_apple_pie and bake into main.py.\n\nClick here for example\nEx: If the input is:\nEve\n10\nthen the output is:\n\nEve buys a pie with 10 apples.\n10 apples are baked into one apple pie.\n\n\nmain.py\n\n""" Your code goes here """\n\ncustomer_name = input()\nnum_apples = int(input())\n\nbuy_apple_pie(customer_name, num_apples)\nbake(num_apples)\npie_shop.py\ndef buy_apple_pie(customer_name, num_apples):\n\tprint(f"{customer_name} buys a pie with {num_apples} apples.")\n\napple_pie.py\ndef bake(num_apples):\n\tprint(f"{num_apples} apples are baked into one apple pie.")\n\n1\n\n2\n\n3\n\nCheck\n\nNext level\n1\n2\n3'

Created file: pie_shop.py with content:
def buy_apple_pie(customer_name, num_apples):
    print(f"{customer_name} buys a pie with {num_apples} apples.")
Created file: apple_pie.py with content:
def bake(num_apples):
    print(f"{num_apples} apples are baked into one apple pie.")
Created file: main.py with content:
from pie_shop import buy_apple_pie
from apple_pie import bake
customer_name = input()
num_apples = int(input())
buy_apple_pie(customer_name, num_apples)
bake(num_apples)
Ran command: python main.py in driver_workspace/challenge_activity_11.3.1_part_1
Eve buys a pie with 10 apples.
10 apples are baked into one apple pie.

Saved answer to ANSWER.md:
# Challenge Activity 11.3.1
1) from pie_shop import buy_apple_pie
from apple_pie import bake
2) Eve buys a pie with 10 apples.
3) 10 apples are baked into one apple pie.



In [ ]:
_ = Problem("scratchwork")
_.create("test.py", 
'''
def fct():
    return "Hello from test.py!"
    print(fct())
''')

_.create("main.py",
         '''
import test
print(test.fct())
         ''')
_.runpy("main.py", cwd=_.folder)

Created file: test.py with content:

def fct():
    return "Hello from test.py!"
    print(fct())

Created file: main.py with content:

import test
print(test.fct())
         
Ran command: python main.py in driver_workspace/scratchwork
Hello from test.py!



In [138]:
'''challenge activity
11.3.1: Importing names directly.
712910.5105864.qx3zqy7

Jump to level 1
Module pie_shop defines buy_apple_pie() and module apple_pie defines bake(). Import the names buy_apple_pie and bake into main.py.

Click here for example
Ex: If the input is:
Eve
10
then the output is:

Eve buys a pie with 10 apples.
10 apples are baked into one apple pie.


main.py

""" Your code goes here """

customer_name = input()
num_apples = int(input())

buy_apple_pie(customer_name, num_apples)
bake(num_apples)
pie_shop.py
def buy_apple_pie(customer_name, num_apples):
	print(f"{customer_name} buys a pie with {num_apples} apples.")
apple_pie.py
def bake(num_apples):
	print(f"{num_apples} apples are baked into one apple pie.")

""" Your code goes here """

customer_name = input()
num_apples = int(input())

buy_apple_pie(customer_name, num_apples)
bake(num_apples)



1

2

3

Check

Next level
1
2
3'''
_ = Problem("challenge activity 11.3.1 part 1")
_.create("pie_shop.py", '''def buy_apple_pie(customer_name, num_apples):
    print(f"{customer_name} buys a pie with {num_apples} apples.")''')
_.create("apple_pie.py", '''def bake(num_apples):
    print(f"{num_apples} apples are baked into one apple pie.")''')
_.create("main.py", '''from pie_shop import buy_apple_pie
from apple_pie import bake
customer_name = input()
num_apples = int(input())
buy_apple_pie(customer_name, num_apples)
bake(num_apples)''')
_.runpy("main.py", inputs=["Eve", "10"])
_.answer('''# Challenge Activity 11.3.1 Part 1
1) from pie_shop import buy_apple_pie
from apple_pie import bake
2) Eve buys a pie with 10 apples.
3) 10 apples are baked into one apple pie.
''')

'challenge activity\n11.3.1: Importing names directly.\n712910.5105864.qx3zqy7\n\nJump to level 1\nModule pie_shop defines buy_apple_pie() and module apple_pie defines bake(). Import the names buy_apple_pie and bake into main.py.\n\nClick here for example\nEx: If the input is:\nEve\n10\nthen the output is:\n\nEve buys a pie with 10 apples.\n10 apples are baked into one apple pie.\n\n\nmain.py\n\n""" Your code goes here """\n\ncustomer_name = input()\nnum_apples = int(input())\n\nbuy_apple_pie(customer_name, num_apples)\nbake(num_apples)\npie_shop.py\ndef buy_apple_pie(customer_name, num_apples):\n\tprint(f"{customer_name} buys a pie with {num_apples} apples.")\napple_pie.py\ndef bake(num_apples):\n\tprint(f"{num_apples} apples are baked into one apple pie.")\n\n""" Your code goes here """\n\ncustomer_name = input()\nnum_apples = int(input())\n\nbuy_apple_pie(customer_name, num_apples)\nbake(num_apples)\n\n\n\n1\n\n2\n\n3\n\nCheck\n\nNext level\n1\n2\n3'

Created file: pie_shop.py with content:
def buy_apple_pie(customer_name, num_apples):
    print(f"{customer_name} buys a pie with {num_apples} apples.")
Created file: apple_pie.py with content:
def bake(num_apples):
    print(f"{num_apples} apples are baked into one apple pie.")
Created file: main.py with content:
from pie_shop import buy_apple_pie
from apple_pie import bake
customer_name = input()
num_apples = int(input())
buy_apple_pie(customer_name, num_apples)
bake(num_apples)
Ran command: python main.py in driver_workspace/challenge_activity_11.3.1_part_1
Eve buys a pie with 10 apples.
10 apples are baked into one apple pie.

Saved answer to ANSWER.md:
# Challenge Activity 11.3.1 Part 1
1) from pie_shop import buy_apple_pie
from apple_pie import bake
2) Eve buys a pie with 10 apples.
3) 10 apples are baked into one apple pie.



In [ ]:
'''challenge activity
11.3.1: Importing names directly.
712910.5105864.qx3zqy7

Jump to level 1
Module airline_desk defines check_in() and check(). Import the names check_in and check into main.py.

Click here for example
Ex: If the input is:
Eve
4
then the output is:

Eve checks in with 4 pieces of luggage.
4 pieces of luggage cost 160 dollars.


main.py

""" Your code goes here """

traveler_name = input()
luggage_quantity = int(input())

check_in(traveler_name, luggage_quantity)
check(luggage_quantity)
airline_desk.py
def check_in(traveler_name, luggage_quantity):
	print(f"{traveler_name} checks in with {luggage_quantity} pieces of luggage.")

def check(luggage_quantity):
	charge = luggage_quantity * 40
	print(f"{luggage_quantity} pieces of luggage cost {charge} dollars.")
1

2

3

Check

Next level'''
_ = Problem("challenge activity 11.3.1 part 2")
_.create("airline_desk.py", '''
def check_in(traveler_name, luggage_quantity):
	print(f"{traveler_name} checks in with {luggage_quantity} pieces of luggage.")

def check(luggage_quantity):
	charge = luggage_quantity * 40
	print(f"{luggage_quantity} pieces of luggage cost {charge} dollars.")''')
_.create("main.py", '''
from airline_desk import check_in, check
traveler_name = input()
luggage_quantity = int(input())

check_in(traveler_name, luggage_quantity)
check(luggage_quantity)''')
_.runpy("main.py", inputs=["Eve", "4"])
_.answer('''# Challenge Activity 11.3.1 Part 2
1) from airline_desk import check_in, check
2) Eve checks in with 4 pieces of luggage.
3) 4 pieces of luggage cost 160 dollars.
''')

'challenge activity\n11.3.1: Importing names directly.\n712910.5105864.qx3zqy7\n\nJump to level 1\nModule airline_desk defines check_in() and check(). Import the names check_in and check into main.py.\n\nClick here for example\nEx: If the input is:\nEve\n4\nthen the output is:\n\nEve checks in with 4 pieces of luggage.\n4 pieces of luggage cost 160 dollars.\n\n\nmain.py\n\n""" Your code goes here """\n\ntraveler_name = input()\nluggage_quantity = int(input())\n\ncheck_in(traveler_name, luggage_quantity)\ncheck(luggage_quantity)\nairline_desk.py\ndef check_in(traveler_name, luggage_quantity):\n\tprint(f"{traveler_name} checks in with {luggage_quantity} pieces of luggage.")\n\ndef check(luggage_quantity):\n\tcharge = luggage_quantity * 40\n\tprint(f"{luggage_quantity} pieces of luggage cost {charge} dollars.")\n1\n\n2\n\n3\n\nCheck\n\nNext level'

Created file: airline_desk.py with content:
def check_in(traveler_name, luggage_quantity):
	print(f"{traveler_name} checks in with {luggage_quantity} pieces of luggage.")

def check(luggage_quantity):
	charge = luggage_quantity * 40
	print(f"{luggage_quantity} pieces of luggage cost {charge} dollars.")
Created file: main.py with content:

from airline_desk import check_in, check
traveler_name = input()
luggage_quantity = int(input())

check_in(traveler_name, luggage_quantity)
check(luggage_quantity)
Ran command: python main.py in driver_workspace/challenge_activity_11.3.1_part_2
Eve checks in with 4 pieces of luggage.
4 pieces of luggage cost 160 dollars.

Saved answer to ANSWER.md:
# Challenge Activity 11.3.1 Part 2
1) from airline_desk import check_in, check
2) Eve checks in with 4 pieces of luggage.
3) 4 pieces of luggage cost 160 dollars.



In [141]:
'''challenge activity
11.3.1: Importing names directly.
712910.5105864.qx3zqy7

Jump to level 1
m_to_cm() is imported from module si_conversion and area() is imported from module tri_shape.

Integers base_in_m and height_in_m are read from input, representing a triangle's base and height in meters. m_to_cm() converts a measurement from meters to centimeters.

Assign area_m2 with the value returned by area() called with base_in_m and height_in_m as the arguments, in that order.
Find the triangle's base and height in centimeters. Then, assign area_cm2 with the value returned by area() called with the triangle's base and height in centimeters as the arguments, in that order.
Click here for example
Ex: If the input is:
5
3
then the output is:

Triangle area is 7.5 m^2 or 75000.0 cm^2.
main.py
from si_conversion import m_to_cm
from tri_shape import area

base_in_m = int(input())
height_in_m = int(input())

""" Your code goes here """

print(f"Triangle area is {area_m2} m^2 or {area_cm2} cm^2.")
si_conversion.py
def m_to_cm(m):
	return m * 100
tri_shape.py
def area(base, height):
	return base * height / 2
'''
_ = Problem("challenge activity 11.3.1 part 3")
_.create("si_conversion.py", '''def m_to_cm(m):
    return m * 100''')
_.create("tri_shape.py", '''def area(base, height):
    return base * height / 2''')
_.create("main.py", '''from si_conversion import m_to_cm
from tri_shape import area
base_in_m = int(input())
height_in_m = int(input())
area_m2 = area(base_in_m, height_in_m)
base_in_cm = m_to_cm(base_in_m)
height_in_cm = m_to_cm(height_in_m)
area_cm2 = area(base_in_cm, height_in_cm)
print(f"Triangle area is {area_m2} m^2 or {area_cm2} cm^2.")''')
_.runpy("main.py", inputs=["5", "3"])
_.answer('''# Challenge Activity 11.3.1 Part 3
1) from si_conversion import m_to_cm
from tri_shape import area
2) area_m2 = area(base_in_m, height_in_m)
base_in_cm = m_to_cm(base_in_m)
height_in_cm = m_to_cm(height_in_m)
area_cm2 = area(base_in_cm, height_in_cm)
3) Triangle area is 7.5 m^2 or 75000.0 cm^2.
''')


'challenge activity\n11.3.1: Importing names directly.\n712910.5105864.qx3zqy7\n\nJump to level 1\nm_to_cm() is imported from module si_conversion and area() is imported from module tri_shape.\n\nIntegers base_in_m and height_in_m are read from input, representing a triangle\'s base and height in meters. m_to_cm() converts a measurement from meters to centimeters.\n\nAssign area_m2 with the value returned by area() called with base_in_m and height_in_m as the arguments, in that order.\nFind the triangle\'s base and height in centimeters. Then, assign area_cm2 with the value returned by area() called with the triangle\'s base and height in centimeters as the arguments, in that order.\nClick here for example\nEx: If the input is:\n5\n3\nthen the output is:\n\nTriangle area is 7.5 m^2 or 75000.0 cm^2.\nmain.py\nfrom si_conversion import m_to_cm\nfrom tri_shape import area\n\nbase_in_m = int(input())\nheight_in_m = int(input())\n\n""" Your code goes here """\n\nprint(f"Triangle area is {ar

Created file: si_conversion.py with content:
def m_to_cm(m):
    return m * 100
Created file: tri_shape.py with content:
def area(base, height):
    return base * height / 2
Created file: main.py with content:
from si_conversion import m_to_cm
from tri_shape import area
base_in_m = int(input())
height_in_m = int(input())
area_m2 = area(base_in_m, height_in_m)
base_in_cm = m_to_cm(base_in_m)
height_in_cm = m_to_cm(height_in_m)
area_cm2 = area(base_in_cm, height_in_cm)
print(f"Triangle area is {area_m2} m^2 or {area_cm2} cm^2.")
Ran command: python main.py in driver_workspace/challenge_activity_11.3.1_part_3
Triangle area is 7.5 m^2 or 75000.0 cm^2.

Saved answer to ANSWER.md:
# Challenge Activity 11.3.1 Part 3
1) from si_conversion import m_to_cm
from tri_shape import area
2) area_m2 = area(base_in_m, height_in_m)
base_in_cm = m_to_cm(base_in_m)
height_in_cm = m_to_cm(height_in_m)
area_cm2 = area(base_in_cm, height_in_cm)
3) Triangle area is 7.5 m^2 or 75000.0 cm^2.




https://learn.zybooks.com/zybook/CPPCS2520NguyenSpring2026/chapter/11/section/4

## 11.4 Executing modules as scripts

An import statement executes the code contained within the imported module. Thus, any statements in the global scope of a module, such as printing or getting user input, will be executed when that module is imported. Execution of those  statements may be an unintended side effect of the import. A programmer wants to treat a Python file as both a script executed by the interpreter and as an importable module. When used as an importable module, the file should not produce side effects when imported.

Consider the following Python file WebSearch.py, which contains functions for returning URLs based on a search term. Executing the file as a script produces the following output:

> **WebSearch.py: Return up to six unique domain names for a given search term.**
> ```python
> # WebSearch.py
> from ddgs import DDGS
> from urllib.parse import urlparse
> 
> def searchTerm(term):
>     print('Searching for "', term, '" ...', sep="")
>     with DDGS() as ddgs:
>         results = [r['href'] for r in ddgs.text(term, max_results=6)]
>     return results
> 
> def stringifyResult(term, domains, domainString):
>     i = 0
>     for url in domains:
>         domain = urlparse(url).netloc
>         if domain and domainString.find(domain) == -1:
>             domainString += "\n" + domain
>             i = i + 1
>     if i == 0:
>         print("No domains", end="")
>     elif i == 1:
>         print("One domain", end="")
>     elif i == 6:
>         print("Search stopped after six domains", end="")
>     else:
>         print(i, "domains", end="")
>     print(' found for search term "', term, '".', sep="")
>     return domainString
> 
> print("Welcome to Web Search")
> term  = input("Enter search term: ")
> result = searchTerm(term)
> resultString = ""
> resultString = stringifyResult(term, result, resultString)
> print(resultString)
> print("\nEnd of Web Search")
> ```
> ```
> Welcome to Web Search
> Enter search term: Funny pictures of cats
> Searching for "Funny pictures of cats" ...
> Search stopped after six domains found for search term "Funny pictures of cats".
> 
> cats.com
> articles.hepper.com
> tenor.com
> unsplash.com
> pixabay.com
> www.rd.com
> 
> End of Web Search
> ```
> Note that the above program imports and uses the ddgs module, which provides functions for fetching URLs using the DuckDuckGo.com search engine. To run the program, the package ddgs (Ex: `pip install ddgs`) must be installed.

If another script imports WebSearch.py to use the searchTerm() function, the statements at the bottom of WebSearch.py will also execute. WebSearchMultiple.py enables the user to enter more than one search term and have results from all search terms returned. However, importing WebSearch.py causes an unwanted prompt for and search of a single term, because that search is in the global scope of WebSearch.py.

> **MultipleWebSearch.py: Importing WebSearch.py causes an unintended search to occur.**
> ```python
> # MultipleWebSearch.py
> import WebSearch as ws  # Causes unintended search
> 
> print("Welcome to Multiple Web Search")
> term = input('Enter search term ("q" to quit): ')
> terms = 0
> resultString = ""
> while term != "q":
>     terms = terms + 1
>     result = ws.searchTerm(term)
>     resultString = ws.stringifyResult(term, result, resultString)
>     term  = input('Enter search term ("q" to quit): ')
> if terms == 0:
>     print("No search terms entered.")
> else:
>     if terms == 1:
>         print("Results for the one search term entered:")
>     else:
>         print("Results for the", terms, "search terms entered:")
>     print(resultString)
> print("\nEnd of Multiple Web Search")
> ```
> ```
> Welcome to Web Search
> Enter search term: Music videos
> Searching for "Music videos" ...
> Search stopped after six domains found for search term "Music videos".
> 
> en.wikipedia.org
> grokipedia.com
> www.youtube.com
> vimeo.com
> music.apple.com
> www.npr.org
> 
> End of Web Search
> Welcome to Multiple Web Search
> Enter search term ("q" to quit): Statue of Liberty
> Searching for "Statue of Liberty" ...
> 5 domains found for search term "Statue of Liberty".
> Enter search term ("q" to quit): Mount Rushmore
> Searching for "Mount Rushmore" ...
> 3 domains found for search term "Mount Rushmore".
> Enter search term ("q" to quit): q
> Results for the 2 search terms entered:
> 
> en.wikipedia.org
> grokipedia.com
> www.nps.gov
> whc.unesco.org
> www.statueofliberty.org
> share.america.gov
> www.worldatlas.com
> www.britannica.com
> 
> End of Multiple Web Search
> ```

A file can better support execution as both a script and an importable module by utilizing the __name__ special variable. __name__ is a global string variable automatically added to every module that contains the name of the module. Ex: my_funcs.__name__ would have the value "my_funcs", and WebSearch.__name__ would have the value "WebSearch". (Note that __name__ has two underscores before name and two underscores after.) However, the value of __name__ for the executing script is always set to "__main__" to differentiate the script from imported modules. The following comparison can be performed:

> **Checking if a file is the executing script or an imported module.**
> ```python
> if __name__ == "__main__":
>     # File executed as a script
> ```

If `if __name__ == "__main__"` is true, then the file is being executed as a script and the branch is taken. Otherwise, the file was imported and thus __name__ is equal to the module name, e.g., "WebSearch".

The contents of the branch typically include a user interface to functions or class definitions within the file. A user can execute the file as a script and interact with the user interface, or another script can import the file just to use the definitions. The WebSearch.py file is modified below to fix the unintentional search.

> **WebSearch.py modified to run as either a script (left) or imported module (right).**
> Each file below is executed as a script.
> | # WebSearch.py from ddgs import DDGS from urllib.parse import urlparse  def searchTerm(term):     # ...  def stringifyResult(term, domains, domainString):     # ...  if __name__ == "__main__":     print("Welcome to Web Search")     term  = input("Enter search term: ")     result = searchTerm(term)     resultString = ""     resultString = stringifyResult(term, result, resultString)     print(resultString)     print("\nEnd of Web Search") | # MultipleWebSearch.py import WebSearch as ws  print("Welcome to Multiple Web Search") term = input('Enter search term ("q" to quit): ') terms = 0 resultString = "" while term != "q":     terms = terms + 1     result = ws.searchTerm(term)     resultString = ws.stringifyResult(term, result, resultString)     term  = input('Enter search term ("q" to quit): ') if terms == 0:     print("No search terms entered.") else:     if terms == 1:         print("Results for the one search term entered:")     else:         print("Results for the", terms, "search terms entered:")     print(resultString) print("\nEnd of Multiple Web Search") |
> | `Welcome to Web Search Enter search term: Music videos Searching for "Music videos" ... Search stopped after six domains found for search term "Music videos".  en.wikipedia.org grokipedia.com www.youtube.com music.apple.com vimeo.com www.npr.org  End of Web Search` | `Welcome to Multiple Web Search Enter search term ("q" to quit): Statue of Liberty Searching for "Statue of Liberty" ... Search stopped after six domains found for search term "Statue of Liberty". Enter search term ("q" to quit): Mount Rushmore Searching for "Mount Rushmore" ... 3 domains found for search term "Mount Rushmore". Enter search term ("q" to quit): q Results for the 2 search terms entered:  en.wikipedia.org grokipedia.com www.nps.gov www.statueofliberty.org whc.unesco.org www.statueoflibertytickets.com share.america.gov www.worldatlas.com www.britannica.com  End of Multiple Web Search` |

The WebSearch.py file has been modified to compare __name__ to "__main__". When the file is executed as a script, a single search request is made and the results are displayed. Executing MultipleWebSearch.py imports WebSearch, which now does not perform the initial search because __name__ is equal to "WebSearch".

### PARTICIPATION ACTIVITY: Executing modules as scripts.

**1.** Importing a module executes the statements contained within the imported module.
Answer: **True**
*A file imported as a module should not contain statements at the global scope that create side effects when importing that module, such as printing output.*

**2.** The value of the __name__ variable of the executing script is always "__main__".
Answer: **True**
*__main__ is added automatically to every module.*

**3.** If a module is imported with the statement `import MyMod`, then MyMod.__name__ is equal to "__main__".
Answer: **False**
*The __name__ variable of any imported module contains the name of the module, thus MyMod.__name__ is equal to "MyMod".*

In [11]:
__='''
F:
11.4 Executing modules as scripts
An import statement executes the code contained within the imported module. Thus, any statements in the global scope of a module, such as printing or getting user input, will be executed when that module is imported. Execution of those statements may be an unintended side effect of the import. A programmer wants to treat a Python file as both a script executed by the interpreter and as an importable module. When used as an importable module, the file should not produce side effects when imported.

Consider the following Python file WebSearch.py, which contains functions for returning URLs based on a search term. Executing the file as a script produces the following output:

Figure 11.4.1: WebSearch.py: Return up to six unique domain names for a given search term.
# WebSearch.py
from ddgs import DDGS
from urllib.parse import urlparse

def searchTerm(term):
    print('Searching for "', term, '" ...', sep="")
    with DDGS() as ddgs:
        results = [r['href'] for r in ddgs.text(term, max_results=6)]
    return results

def stringifyResult(term, domains, domainString):
    i = 0
    for url in domains:
        domain = urlparse(url).netloc
        if domain and domainString.find(domain) == -1:
            domainString += "\n" + domain
            i = i + 1
    if i == 0:
        print("No domains", end="")
    elif i == 1:
        print("One domain", end="")
    elif i == 6:
        print("Search stopped after six domains", end="")
    else:
        print(i, "domains", end="")
    print(' found for search term "', term, '".', sep="")
    return domainString

print("Welcome to Web Search")
term  = input("Enter search term: ")
result = searchTerm(term)
resultString = ""
resultString = stringifyResult(term, result, resultString)
print(resultString)
print("\nEnd of Web Search")
Welcome to Web Search
Enter search term: Funny pictures of cats
Searching for "Funny pictures of cats" ...
Search stopped after six domains found for search term "Funny pictures of cats".

cats.com
articles.hepper.com
tenor.com
unsplash.com
pixabay.com
www.rd.com

End of Web Search
Note that the above program imports and uses the ddgs module, which provides functions for fetching URLs using the DuckDuckGo.com search engine. To run the program, the package ddgs (Ex: pip install ddgs) must be installed.


Feedback?
If another script imports WebSearch.py to use the searchTerm() function, the statements at the bottom of WebSearch.py will also execute. WebSearchMultiple.py enables the user to enter more than one search term and have results from all search terms returned. However, importing WebSearch.py causes an unwanted prompt for and search of a single term, because that search is in the global scope of WebSearch.py.

Figure 11.4.2: MultipleWebSearch.py: Importing WebSearch.py causes an unintended search to occur.
# MultipleWebSearch.py
import WebSearch as ws  # Causes unintended search

print("Welcome to Multiple Web Search")
term = input('Enter search term ("q" to quit): ')
terms = 0
resultString = ""
while term != "q":
    terms = terms + 1
    result = ws.searchTerm(term)
    resultString = ws.stringifyResult(term, result, resultString)
    term  = input('Enter search term ("q" to quit): ')
if terms == 0:
    print("No search terms entered.")
else:
    if terms == 1:
        print("Results for the one search term entered:")
    else:
        print("Results for the", terms, "search terms entered:")
    print(resultString)
print("\nEnd of Multiple Web Search")
Welcome to Web Search
Enter search term: Music videos
Searching for "Music videos" ...
Search stopped after six domains found for search term "Music videos".

en.wikipedia.org
grokipedia.com
www.youtube.com
vimeo.com
music.apple.com
www.npr.org

End of Web Search
Welcome to Multiple Web Search
Enter search term ("q" to quit): Statue of Liberty
Searching for "Statue of Liberty" ...
5 domains found for search term "Statue of Liberty".
Enter search term ("q" to quit): Mount Rushmore
Searching for "Mount Rushmore" ...
3 domains found for search term "Mount Rushmore".
Enter search term ("q" to quit): q
Results for the 2 search terms entered:

en.wikipedia.org
grokipedia.com
www.nps.gov
whc.unesco.org
www.statueofliberty.org
share.america.gov
www.worldatlas.com
www.britannica.com

End of Multiple Web Search

Feedback?
A file can better support execution as both a script and an importable module by utilizing the __name__ special variable. __name__ is a global string variable automatically added to every module that contains the name of the module. Ex: my_funcs.__name__ would have the value "my_funcs", and WebSearch.__name__ would have the value "WebSearch". (Note that __name__ has two underscores before name and two underscores after.) However, the value of __name__ for the executing script is always set to "__main__" to differentiate the script from imported modules. The following comparison can be performed:

Figure 11.4.3: Checking if a file is the executing script or an imported module.
if __name__ == "__main__":
    # File executed as a script

Feedback?
If if __name__ == "__main__" is true, then the file is being executed as a script and the branch is taken. Otherwise, the file was imported and thus __name__ is equal to the module name, e.g., "WebSearch".

The contents of the branch typically include a user interface to functions or class definitions within the file. A user can execute the file as a script and interact with the user interface, or another script can import the file just to use the definitions. The WebSearch.py file is modified below to fix the unintentional search.

Figure 11.4.4: WebSearch.py modified to run as either a script (left) or imported module (right).
Each file below is executed as a script.

WebSearch.py
# WebSearch.py
from ddgs import DDGS
from urllib.parse import urlparse

def searchTerm(term):
    # ...

def stringifyResult(term, domains, domainString):
    # ...

if __name__ == "__main__":
    print("Welcome to Web Search")
    term  = input("Enter search term: ")
    result = searchTerm(term)
    resultString = ""
    resultString = stringifyResult(term, result, resultString)
    print(resultString)
    print("\nEnd of Web Search")
MultipleWebSearch.py
# MultipleWebSearch.py
import WebSearch as ws

print("Welcome to Multiple Web Search")
term = input('Enter search term ("q" to quit): ')
terms = 0
resultString = ""
while term != "q":
    terms = terms + 1
    result = ws.searchTerm(term)
    resultString = ws.stringifyResult(term, result, resultString)
    term  = input('Enter search term ("q" to quit): ')
if terms == 0:
    print("No search terms entered.")
else:
    if terms == 1:
        print("Results for the one search term entered:")
    else:
        print("Results for the", terms, "search terms entered:")
    print(resultString)
print("\nEnd of Multiple Web Search")
Welcome to Web Search
Enter search term: Music videos
Searching for "Music videos" ...
Search stopped after six domains found for search term "Music videos".

en.wikipedia.org
grokipedia.com
www.youtube.com
music.apple.com
vimeo.com
www.npr.org

End of Web Search
Welcome to Multiple Web Search
Enter search term ("q" to quit): Statue of Liberty
Searching for "Statue of Liberty" ...
Search stopped after six domains found for search term "Statue of Liberty".
Enter search term ("q" to quit): Mount Rushmore
Searching for "Mount Rushmore" ...
3 domains found for search term "Mount Rushmore".
Enter search term ("q" to quit): q
Results for the 2 search terms entered:

en.wikipedia.org
grokipedia.com
www.nps.gov
www.statueofliberty.org
whc.unesco.org
www.statueoflibertytickets.com
share.america.gov
www.worldatlas.com
www.britannica.com

End of Multiple Web Search

Feedback?
The WebSearch.py file has been modified to compare __name__ to "__main__". When the file is executed as a script, a single search request is made and the results are displayed. Executing MultipleWebSearch.py imports WebSearch, which now does not perform the initial search because __name__ is equal to "WebSearch".

participation activity
11.4.1: Executing modules as scripts.
1)
Importing a module executes the statements contained within the imported module.
2)
The value of the __name__ variable of the executing script is always "__main__".
3)
If a module is imported with the statement import MyMod, then MyMod.__name__ is equal to "__main__".

Feedback?
How was this section?

|


Provide section feedback
F::QUESTION.md

F:
1) Importing a module executes the statements contained within the imported module.
True. 
When a module is imported, the code in the global scope of that module is executed. This can lead to unintended side effects if the module contains statements that perform actions (e.g., print statements, input statements) when imported
2) The value of the __name__ variable of the executing script is always "__main__".
True.
The __name__ variable is a special built-in variable in Python that is automatically set to "__main__" when the file is executed as the main program. If the file is imported as a module, __name__ is set to the name of the module instead.
3) If a module is imported with the statement import MyMod, then MyMod.__name__ is equal to "__main__".
False.
When a module is imported, the __name__ variable of that module is set to the name of the module (e.g., "MyMod"), not "__main__". The __name__ variable is only set to "__main__" for the file that is being executed as the main program, not for imported modules.

# Participation Activity 11.4.1
1) True
2) True
3) False
F::ANSWER.md

'''

_= Problem("Participation Activity 11.4.1")
_.parse(__)

Created file: QUESTION.md with content:
11.4 Executing modules as scripts
An import statement executes the code contained within the imported module. Thus, any statements in the global scope of a module, such as printing or getting user input, will be executed when that module is imported. Execution of those statements may be an unintended side effect of the import. A programmer wants to treat a Python file as both a script executed by the interpreter and as an importable module. When used as an importable module, the file should not produce side effects when imported.

Consider the following Python file WebSearch.py, which contains functions for returning URLs based on a search term. Executing the file as a script produces the following output:

Figure 11.4.1: WebSearch.py: Return up to six unique domain names for a given search term.
# WebSearch.py
from ddgs import DDGS
from urllib.parse import urlparse

def searchTerm(term):
    print('Searching for "', term, '" ...', sep="")
    wi

Problem(name='Participation Activity 11.4.1', files=2, folder='driver_workspace/participation_activity_11.4.1')

In [ ]:
#is this easir for you or harder?
#I can do it either way, but I think this is easier for me to write and read than the previous format.

#you sure?
#Yes, I think this format is more straightforward for me to write and read. The previous format with the question and answer sections was a bit more complex to structure and read through. This new format allows me to directly address each question and provide the answer in a clear and concise manner.

11.5 Reloading modules

https://learn.zybooks.com/zybook/CPPCS2520NguyenSpring2026/chapter/11/section/5

In [19]:


__='''F:
## 11.5 Reloading modules

Sometimes a Python program imports a module, but then the source code of the imported module needs to be changed. Since modules are executed only once when imported, changing the module's source does not affect the running Python instance. Instead of restarting the entire Python program, the reload() function can be used to reload and re-execute the changed module. The reload() function is located in the importlib standard library module.

Consider the following module, which can send an email using a Google gmail account:

> **send_gmail.py: Sends a single email through gmail.**
> ```python
> import smtplib
> from email.mime.text import MIMEText
> 
> header = "Hello. This is an automated email.\n\n"
> 
> def send(subject, to, frm, text):
>     # The message to send
>     msg = MIMEText(header + text)
>     msg["Subject"] = subject
>     msg["To"] = to
>     msg["From"] = frm
> 
>     # Connect to gmail's email server and send
>     s = smtplib.SMTP("smtp.gmail.com", port=587)
>     s.ehlo()
>     s.starttls()
>     s.login(user=frm, password="password")
>     s.sendmail(frm, [to], msg.as_string())
>     s.quit()
> 
> if __name__ == "__main__":
>     send(
>         subject="A coupon for you!",
>         to="billgates@microsoft.com",
>         frm="JohnnysHotDogs1@gmail.com",
>         text="Enjoy!")
> ```
> ```
> To: billgates@microsoft.com
> From: JohnnysHotDogs1@gmail.com
> Subject: A coupon for you!
> 
> Hello. This is an automated email.
> 
> Enjoy!
> ```

The send_coupons.py script below imports send_gmail.py as a module, using the send function to deliver important messages to customers.

> **send_coupons.py: Automates emails to loyal customers.**
> ```python
> import os
> from importlib import reload
> import send_gmail
> 
> mod_time = os.path.getmtime(send_gmail.__file__)
> 
> emails = [  # Could be large list or stored in file
>     "billgates@microsoft.com",
>     "president@whitehouse.gov",
>     "benedictxvi@vatican.va"
> ]
> 
> my_email = "JohnnysHotDogs1@gmail.com"
> subject = "A coupon for you!"
> text = ("As a loyal customer of Johnny's HotDogs, "
>         "here is a coupon for 1 free bratwurst!")
> 
> for addr in emails:
>     send_gmail.send(subject, addr, my_email, text)
> 
>     # Check if file has been modified
>     last_mod = os.path.getmtime(send_gmail.__file__) 
>     if last_mod > mod_time:
>         mod_time = last_mod
>         reload(send_gmail)
> ```

If thousands of emails are being sent, the program should not be stopped because rerunning the program could cause duplicate emails to be sent to some users, and Johnny's HotDogs might annoy their customers. If Johnny wants to change the content of the header string in the send_gmail module without stopping the program, then the variable's value in send_gmail.py's *source code* can be updated and reloaded.

When send_coupons.py imports send_gmail, a global variable mod_time stores the time when send_gmail.py was last modified, using the os.path.getmtime() function. The __file__ special name contains the path to a module in the computer file system, e.g., the value of send_gmail.__file__ might be "C:\\Users\\Johnny\\send_gmail.py". A comparison is made to the original modification time at the end of the for loop &ndash; if the modification time is greater than the original, then the module's source code has been updated and the module should be reloaded.

Modifying the header string in send_gmail.py to "This is an important message from Johnny" while the program is running causes the module to be reloaded, which alters the contents of the emails.

> **Modifying send_gmail.py while the program is running updates the email contents.**
> ```python
> import smtplib
> from email.mime.text import MIMEText
> 
> header = "This is an important message from Johnny!"
> 
> def send(subject, to, frm, txt):
>     # ...
> ```
> ```
> To: president@whitehouse.gov
> From: JohnnysHotDogs1@gmail.com
> Subject: A coupon for you!
> 
> This is an important message from Johnny!
> 
> As a loyal customer of Johnny's HotDogs, 
> here is a coupon for 1 free bratwurst!
> ```

The reload function reloads a module in place. When reload(send_gmail) returns, the namespace of the send_gmail module will contain updated definitions. The call to send_gmail.send() still accesses the same send_gmail module object, but the definition of send() will have been updated.

Importing attributes directly using "from", and then reloading the corresponding module, will *not* update the imported attributes.

> **Reloading modules doesn't affect attributes imported using 'from'.**
> | from importlib import reload import send_gmail from send_gmail import header  print(f"Original value of header: {header}")  print("\n(---- send_gmail.py source code edited ----)")  print("\nReloading send_gmail\n") reload(send_gmail)  print(f"header: {header}") print(f"send_gmail.header: {send_gmail.header}") |
> | `Original value of header: Hello. This is an automated email.  (---- send_gmail.py edited ----)  Reloading send_gmail.  header: Hello. This is an automated email. send_gmail.header: Hello from Johnny's Hotdogs!` |

Reloading modules is typically useful in long-running programs, when restarting and initializing the entire program may be an expensive operation. A common scenario is a web server that is communicating with multiple clients on the internet. Instead of restarting the server and disconnecting all of the clients, a single module can be reloaded dynamically as the server runs.

### PARTICIPATION ACTIVITY: Reloading modules.

**1.** Modules cannot be reloaded if they have already been imported.
Answer: **False**
*The purpose of the reload function is to reload modules that have changed since importation.*

**2.** The reload function modifies a module in place.
Answer: **True**
*reload() also returns the module but can be ignored.*

**3.** Reloading a module is useful when restarting a program is prohibitively costly.
Answer: **True**
*Reloading is often done in server programs, where shutting down a server may disconnect clients.*
F::QUESTION.md
F:
# Participation Activity 11.5.1
1) Modules cannot be reloaded if they have already been imported.
2) The reload function modifies a module in place.
3) Reloading a module is useful when restarting a program is prohibitively costly.
F::ANSWER.md'''
_ = Problem("Participation Activity 11.5.1")
_.parse(__)

Created file: QUESTION.md with content:
## 11.5 Reloading modules

Sometimes a Python program imports a module, but then the source code of the imported module needs to be changed. Since modules are executed only once when imported, changing the module's source does not affect the running Python instance. Instead of restarting the entire Python program, the reload() function can be used to reload and re-execute the changed module. The reload() function is located in the importlib standard library module.

Consider the following module, which can send an email using a Google gmail account:

> **send_gmail.py: Sends a single email through gmail.**
> ```python
> import smtplib
> from email.mime.text import MIMEText
> 
> header = "Hello. This is an automated email.

"
> 
> def send(subject, to, frm, text):
>     # The message to send
>     msg = MIMEText(header + text)
>     msg["Subject"] = subject
>     msg["To"] = to
>     msg["From"] = frm
> 
>     # Connect to gmail's email server and

Problem(name='Participation Activity 11.5.1', files=2, folder='driver_workspace/participation_activity_11.5.1')

https://learn.zybooks.com/zybook/CPPCS2520NguyenSpring2026/chapter/11/section/6

## 11.6 Packages

Instead of importing a single module at a time, an entire directory of modules can be imported all at once. A package is a directory that, when imported, gives access to all of the modules stored in the directory. Large projects are often organized using packages to group related modules.

### PARTICIPATION ACTIVITY: Packages group related modules together.

Static figure:
Begin Python code 1:
import sound.music
import sound.effects

sound.music.play_sound()
sound.effects.play_whoosh()
# ...
End Python code.

Begin Python code 2:
import game.sound
import game.graphics

game.sound.speech.talk()
game.graphics.terrain.draw_gnd()
# ...
End Python code.

A tree is shown with the Game package as the root. Game has subpackages Sound and Graphics. Sound has modules music.py, effects.py, and speech.py. Graphics has modules models.py, textures.py, and terrain.py.

Step 1: Packages such as "sound" contain subpackages, such as "music" and "effects". Once subpackages are imported, modules and definitions in the subpackages are reached via dot notation. Only the Sound package with the music.py, effects.py, and speech.py modules is shown and the first program begins to execute. Importing sound.music highlights the Sound package and music.py. Importing sound.effects highlights the Sound package and effects.py.

Step 2:The "game" package contains subpackages "sound" and "graphics". The full package tree is shown and the second program begins to execute. The line "game.sound.speech.talk()" executes and Game, Sound, and speech.py are highlighted. The line "game.graphics.terrain.draw_gnd()" executes and Game, Graphics, and terrain.py are highlighted.

To import a package, a programmer writes an import statement and gives the name of the directory where the package is located. To indicate that a directory is a package, the directory must include a file called __init__.py. The _init_.py file often is empty, but may include import statements that import subpackages or modules. The interpreter searches for the package in the directories listed in sys.path.

Consider the following directory structure. A package ASCIIArt contains a canvas module, as well as the subpackages' figures and buildings.

> **Directory structure.**
> ```
> draw_scene.py            Script that imports ASCIIArt package
> ASCIIArt\                     Top-level package
>         __init__.py
>         canvas.py
>         figures\              Subpackage for figures art
>                __init__.py
>                man.py
>                cow.py
>                ...
>         buildings\            Subpackage for buildings art
>                __init__.py
>                barn.py
>                house.py
>                ...
> ```

The draw_scene.py script can import the ASCIIArt package using the following statement:

> **Importing the ASCIIArt package.**
> ```python
> import ASCIIArt  # import ASCIIArt package
> ```

Specific modules or subpackages can be imported individually by specifying the path to the item, using periods in the import name. References to names within the imported module require that the entire path be specified.

> **Importing the canvas module.**
> ```python
> import ASCIIArt.canvas  # imports the canvas.py module
> 
> ASCIIArt.canvas.draw_canvas()  # Definitions in canvas.py have full name specified
> ```

The *from* technique of importing also works with packages, allowing individual modules or subpackages to be directly imported into the global namespace. A benefit of this method is that higher-level package names need not be specified.

> **Import cow module from figures subpackage.**
> ```python
> from ASCIIArt.figures import cow  # import cow module
> 
> cow.draw()  # Can omit ASCIIArt.figures prefix
> ```

Even individual names from a module can be imported, making that name directly available.

> **Import the draw function from the cow module.**
> ```python
> from ASCIIArt.figures.cow import draw  # import draw() function
> 
> draw()  # Can omit ASCIIArt.figures.cow
> ```

When using syntax such as "import y.z", the last item z must be a package, a module, or a subpackage. In contrast, when using "from x.y import z", the last item, z, can also be a name from y, such as a function, class, or global variable.

Using packages helps to avoid module name collisions. Ex: Consider if another package called 3DGraphics also contained a module called canvas.py. Though both modules share a name, they are differentiated by the package that contains them, that is, ASCIIArt.canvas is different from 3DGraphics.canvas. A programmer should take care when using the *from* technique of importing. A common error is to overwrite an imported module with another package's identically named module.

### PARTICIPATION ACTIVITY: Importing packages.

**1.** import Write a statement to import the figures subpackage.
Answer: ASCIIArt.figures
*Hint: Use periods to specify the path to the item to import.*
*The figure's subpackage is imported.*

**2.** import Write a statement to import the cow module.
Answer: ASCIIArt.figures.cow
*Hint: Use periods to specify the path to the item to import.*
*The cow module is imported.*

**3.** from ASCIIArt.buildings.house import draw Write a statement that calls the draw() function of the imported house module.
Answer: draw()
*Hint: Use the full name when calling draw().*
*Importing a module's function using "from" makes that function directly callable; module and package dot notation is omitted.*

**4.** Write a statement that imports the barn module directly using the "from" technique of importing.
Answer: from ASCIIArt.buildings import barn
*Hint: From? Import?*
*Periods are used to specify the path to the desired module.*

### CHALLENGE ACTIVITY: Using packages.

In [29]:
__=r'''
F:

11.6 Packages
Instead of importing a single module at a time, an entire directory of modules can be imported all at once. A package is a directory that, when imported, gives access to all of the modules stored in the directory. Large projects are often organized using packages to group related modules.

participation activity
11.6.1: Packages group related modules together.


1

2

The "game" package contains subpackages "sound" and "graphics".
import sound.music
import sound.effects

sound.music.play_sound()
sound.effects.play_whoosh()
# ...
Soundmusic.pyeffects.pyspeech.py
import game.sound
import game.graphics

game.sound.speech.talk()
game.graphics.terrain.draw_gnd()
# ...
GameGraphicsmodels.pytextures.pyterrain.py
Static figure: Begin Python code 1: import sound.music import sound.effects sound.music.play_sound() sound.effects.play_whoosh() # ... End Python code. Begin Python code 2: import game.sound import game.graphics game.sound.speech.talk() game.graphics.terrain.draw_gnd() # ... End Python code. A tree is shown with the Game package as the root. Game has subpackages Sound and Graphics. Sound has modules music.py, effects.py, and speech.py. Graphics has modules models.py, textures.py, and terrain.py. Step 1: Packages such as "sound" contain subpackages, such as "music" and "effects". Once subpackages are imported, modules and definitions in the subpackages are reached via dot notation. Only the Sound package with the music.py, effects.py, and speech.py modules is shown and the first program begins to execute. Importing sound.music highlights the Sound package and music.py. Importing sound.effects highlights the Sound package and effects.py. Step 2:The "game" package contains subpackages "sound" and "graphics". The full package tree is shown and the second program begins to execute. The line "game.sound.speech.talk()" executes and Game, Sound, and speech.py are highlighted. The line "game.graphics.terrain.draw_gnd()" executes and Game, Graphics, and terrain.py are highlighted.

Captions
Playing step 2: The "game" package contains subpackages "sound" and "graphics".

Feedback?
To import a package, a programmer writes an import statement and gives the name of the directory where the package is located. To indicate that a directory is a package, the directory must include a file called __init__.py. The _init_.py file often is empty, but may include import statements that import subpackages or modules. The interpreter searches for the package in the directories listed in sys.path.

Consider the following directory structure. A package ASCIIArt contains a canvas module, as well as the subpackages' figures and buildings.

Figure 11.6.1: Directory structure.
draw_scene.py            Script that imports ASCIIArt package
ASCIIArt\                     Top-level package
        __init__.py
        canvas.py
        figures\              Subpackage for figures art
               __init__.py
               man.py
               cow.py
               ...
        buildings\            Subpackage for buildings art
               __init__.py
               barn.py
               house.py
               ...

Feedback?
The draw_scene.py script can import the ASCIIArt package using the following statement:

Figure 11.6.2: Importing the ASCIIArt package.
import ASCIIArt  # import ASCIIArt package

Feedback?
Specific modules or subpackages can be imported individually by specifying the path to the item, using periods in the import name. References to names within the imported module require that the entire path be specified.

Figure 11.6.3: Importing the canvas module.
import ASCIIArt.canvas  # imports the canvas.py module

ASCIIArt.canvas.draw_canvas()  # Definitions in canvas.py have full name specified

Feedback?
The from technique of importing also works with packages, allowing individual modules or subpackages to be directly imported into the global namespace. A benefit of this method is that higher-level package names need not be specified.

Figure 11.6.4: Import cow module from figures subpackage.
from ASCIIArt.figures import cow  # import cow module

cow.draw()  # Can omit ASCIIArt.figures prefix

Feedback?
Even individual names from a module can be imported, making that name directly available.

Figure 11.6.5: Import the draw function from the cow module.
from ASCIIArt.figures.cow import draw  # import draw() function

draw()  # Can omit ASCIIArt.figures.cow

Feedback?
When using syntax such as "import y.z", the last item z must be a package, a module, or a subpackage. In contrast, when using "from x.y import z", the last item, z, can also be a name from y, such as a function, class, or global variable.

Using packages helps to avoid module name collisions. Ex: Consider if another package called 3DGraphics also contained a module called canvas.py. Though both modules share a name, they are differentiated by the package that contains them, that is, ASCIIArt.canvas is different from 3DGraphics.canvas. A programmer should take care when using the from technique of importing. A common error is to overwrite an imported module with another package's identically named module.

participation activity
11.6.2: Importing packages.
Consider the directory structure of the ASCIIArt package above.

1)
Write a statement to import the figures subpackage.
import 

Check

Show answer
2)
Write a statement to import the cow module.
import 

Check

Show answer
3)
Write a statement that calls the draw() function of the imported house module.
from ASCIIArt.buildings.house import draw                                 



Check

Show answer
4)
Write a statement that imports the barn module directly using the "from" technique of importing.

Check

Show answer

Feedback?
F::QUESTION.md


F:
# Participation Activity 11.6.2
1) Write a statement to import the figures subpackage.
import ASCIIArt.figures
2) Write a statement to import the cow module.
import ASCIIArt.figures.cow
3) Write a statement that calls the draw() function of the imported house module.
from ASCIIArt.buildings.house import draw
draw()
4) Write a statement that imports the barn module directly using the "from" technique of importing.
from ASCIIArt.buildings import barn
barn.draw()
F::ANSWER.md
F:
from ast import main

import ASCIIArt.figures
import ASCIIArt.figures.cow
from ASCIIArt.buildings.house import draw
draw()
from ASCIIArt.buildings import barn
barn.draw()
F::main.py
R: main.py
'''
_= Problem("Participation Activity 11.6.2")
_.parse(__)




Error: Traceback (most recent call last):
  File "/workspaces/cs2520-study/zybooks notebooks/C10/driver_workspace/participation_activity_11.6.2/main.py", line 3, in <module>
    import ASCIIArt.figures
ModuleNotFoundError: No module named 'ASCIIArt'



Problem(name='Participation Activity 11.6.2', files=3, folder='driver_workspace/participation_activity_11.6.2')

In [31]:
__='''
F:
challenge activity
11.6.1: Using packages.
712910.5105864.qx3zqy7
Jump to level 1
The directory structure of the ASCIIArt package is given. Complete the statement to import the transports subpackage.ASCIIArt\
        __init__.py
        canvas.py
        transports\
               __init__.py
               taxi.py
               bicycle.py

        clothes\
               __init__.py
               hat.py
               socks.pyimport
Ex: ASCIIArt.clothes
1
2
3
4
Check
Next
1
2
3
4

Feedback?
F::QUESTION.md
F:
# Challenge Activity 11.6.1 Part 1
1) The directory structure of the ASCIIArt package is given. Complete the statement to import the transports subpackage.
import ASCIIArt.transports
2) Write a statement to import the taxi module.
F::ANSWER.md

'''
_= Problem("Challenge Activity 11.6.1 Part 1")
_.parse(__)

Problem(name='Challenge Activity 11.6.1 Part 1', files=2, folder='driver_workspace/challenge_activity_11.6.1_part_1')

In [2]:
a='''H:
ASCIIArt\
        __init__.py
        canvas.py
        transports\
               __init__.py
               taxi.py
               bicycle.py

        clothes\
               __init__.py
               hat.py
               socks.py
'''
# i made it upside fown! i also made it able to go both ways! :D also i made it accept FF:
# i can do it either way, but i think this is easier for me to write and read than the previous format.

#we wil see!

In [ ]:
###########
__='''
challenge activity
11.6.1: Using packages.
712910.5105864.qx3zqy7
Jump to level 1
The directory structure of the ASCIIArt package is given. Complete the statement to import the shirt module directly using the "from" technique of importing.ASCIIArt\
        __init__.py
        canvas.py
        clothes\
               __init__.py
               shoes.py
               shirt.py

        animals\
               __init__.py
               turtle.py
               dog.pyfromimport
ASCIIArt.clothes
shirt
1
2
3
4
Check
Next
Correct Expected: ASCIIArt.clothes, shirt

The path to the shirt module is ASCIIArt.clothes.shirt. Using the "from" technique, from ASCIIArt.clothes import shirt imports the shirt module directly.
1
2
3
4

Feedback?
F::QUESTION.md
'''
_= Problem("Challenge Activity 11.6.1 Part 2")

__+='''F:
# Challenge Activity 11.6.1 Part 2
1) The directory structure of the ASCIIArt package is given. Complete the statement to import the shirt module directly using the "from" technique of importing.
from ASCIIArt.clothes import shirt
F::ANSWER.md
'''
_.parse(__)
###########
#awsomesauce
#

Created file: QUESTION.md with content:
challenge activity
11.6.1: Using packages.
712910.5105864.qx3zqy7
Jump to level 1
The directory structure of the ASCIIArt package is given. Complete the statement to import the shirt module directly using the "from" technique of importing.ASCIIArt        __init__.py
        canvas.py
        clothes               __init__.py
               shoes.py
               shirt.py
        animals               __init__.py
               turtle.py
               dog.pyfromimport
ASCIIArt.clothes
shirt
1
2
3
4
Check
Next
Correct Expected: ASCIIArt.clothes, shirt
The path to the shirt module is ASCIIArt.clothes.shirt. Using the "from" technique, from ASCIIArt.clothes import shirt imports the shirt module directly.
1
2
3
4
Feedback?
Created file: ANSWER.md with content:
# Challenge Activity 11.6.1 Part 2
1) The directory structure of the ASCIIArt package is given. Complete the statement to import the shirt module directly using the "from" technique of importi

Problem(name='Challenge Activity 11.6.1 Part 2', files=2, folder='driver_workspace/challenge_activity_11.6.1_part_2')